---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---

# Notebook 15 — Geological Analysis of DT Predictions: Hypothesis Testing

**Objective:** Investigate geological and petrophysical factors that explain
variability in XGBoost DT prediction quality across 27 LOWO wells.

**Hypotheses tested:**
- H1: Depth (burial compaction and diagenesis)
- H2: DT outside training domain (response extrapolation)
- H3: Features outside training domain (feature extrapolation)
- H4: Geological formation (depositional environment)
- H5: Lithology (rock type)
- H6: Layer thickness (vertical resolution)
- H7: Well directionality (inclination/deviation)
- H8: Petrophysical regime (RHOB × NPHI × DT domain)

**Required data:**
- `data/processed/wells_iqr_with_lithology.csv`
- `results/xgboost/predictions/lowo_xgboost_predictions.csv`
- `results/xgboost/metrics/lowo_xgboost_results.csv`
- `data/formations.csv`

## 1. Setup and Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib import patheffects
from matplotlib.lines import Line2D
from scipy.stats import spearmanr, pearsonr, mannwhitneyu
from scipy import stats as scipy_stats
from scipy.interpolate import interp1d
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from adjustText import adjust_text
from sonic_ml_utils import (
    plot_well_petrophysical_log,
    plot_petrophysical_diagnosis_panel,
    FORMATION_COLORS,
    LITHO_COLORS,
    FALLBACK_COLOR,
)

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')


In [ ]:
# Paths
DATA_PATH       = Path('../data/processed/wells_iqr_with_lithology.csv')
PRED_PATH       = Path('../results/xgboost/predictions/lowo_xgboost_predictions.csv')
METRICS_PATH    = Path('../results/xgboost/metrics/lowo_xgboost_results.csv')
FORMATIONS_PATH = Path('../data/formations.csv')
FIGURES_PATH    = Path('../results/comparison/figures/')
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

DIRECTIONAL_PATH = Path('../data/raw/directionals/directional_wells.csv')
GEOGRAPHIC_PATH  = Path('../data/raw/well_coordinates.csv')

MODEL_NAME = 'XGBoost'

# # Formation color palette
# FORMATION_COLORS = {
#     'FM. CALUMBI'      : '#2980B9',
#     'FM. COTINGUIBA'   : '#27AE60',
#     'FM. RIACHUELO'    : '#E67E22',
#     'FM. MURIBECA'     : '#8E44AD',
#     'FM. COQUEIRO SECO': '#E74C3C',
#     'out_of_range'     : '#BDC3C7',
# }

def lith_color(lith):
    """Returns the lithology color."""
    if pd.isna(lith): return FALLBACK_COLOR
    lith_lower = str(lith).lower().strip()
    if lith_lower in LITHO_COLORS: return LITHO_COLORS[lith_lower]
    for key in LITHO_COLORS:
        if key in lith_lower: return LITHO_COLORS[key]
    return FALLBACK_COLOR

def representativity_label(n_samples, n_wells):
    if n_samples >= 5000 and n_wells >= 5: return '✅ High'
    if n_samples >= 500  and n_wells >= 3: return '⚠️ Moderate'
    if n_samples >= 100  and n_wells >= 2: return '🔶 Low'
    return '❌ Very Low'

# Analysis constants
DEPTH_BIN_SIZE = 200   # 200m bins for depth analysis

print(f'Mapped lithologies: {list(LITHO_COLORS.keys())}')

## 2. Data Loading and Preparation

In [ ]:
# Load files
df_main  = pd.read_csv(DATA_PATH)
df_pred  = pd.read_csv(PRED_PATH)
df_metr  = pd.read_csv(METRICS_PATH)
df_form  = pd.read_csv(FORMATIONS_PATH, sep=';')

print(f'Main dataset      : {len(df_main):,} samples, {df_main["Well_Name"].nunique()} wells')
print(f'Predictions       : {len(df_pred):,} samples, {df_pred["Well_Name"].nunique()} wells')
print(f'LOWO metrics      : {len(df_metr)} wells')
print(f'Formation intervals: {len(df_form)} records')

print('\nAvailable Lithologies (main dataset):')
print(df_main['Lithology'].value_counts())

if 'Formation' in df_main.columns:
    print('\nFormation column present in the dataset')
    print(df_main['Formation'].value_counts())
else:
    print('\nFormation column not found — it will be assigned in the merge cell')


### 2.1 Geographic and Directional Data

Loads and consolidates location and directionality data for each well.
Geographic data should be exported from QGIS as a CSV with columns:
`Well_Name, Latitude, Longitude, X_UTM, Y_UTM`.

Directional data is generated by **Notebook 13**.


In [ ]:
# Directional data
if DIRECTIONAL_PATH.exists():
    df_dir = pd.read_csv(DIRECTIONAL_PATH)

    # Per-well summary: max inclination, final MD, final TVD, horizontal displacement
    df_dir_summary = df_dir.groupby('Well_Name').agg(
        Max_INCL        = ('INCL',        'max'),
        MD_final        = ('MD',          'max'),
        TVD_final       = ('TVD',         'max'),
        NS_final        = ('NS',          'last'),
        EW_final        = ('EW',          'last'),
        Max_MD_TVD_diff = ('MD_TVD_diff', 'max'),
        Pct_Deviated    = ('Is_Deviated', 'mean'),
    ).reset_index()

    df_dir_summary['Desl_H']       = np.sqrt(
        df_dir_summary['NS_final']**2 + df_dir_summary['EW_final']**2
    ).round(1)
    df_dir_summary['Pct_Deviated'] = (df_dir_summary['Pct_Deviated'] * 100).round(1)
    df_dir_summary['Dir_Category'] = df_dir_summary['Max_INCL'].apply(
        lambda v: 'Directional (>20°)' if v > 20
        else ('Slight deviation (5–20°)' if v > 5 else 'Vertical (≤5°)')
    )
    print(f'Wells with directional data: {df_dir_summary["Well_Name"].nunique()} wells')
    print(df_dir_summary[['Well_Name','Max_INCL','MD_final','TVD_final',
                           'Desl_H','Dir_Category']].to_string(index=False))
else:
    print(f'{DIRECTIONAL_PATH} not found.')
    print('   Run Notebook 13 first.')
    df_dir_summary = pd.DataFrame()

# Geographic data
if GEOGRAPHIC_PATH.exists():
    df_geo = pd.read_csv(GEOGRAPHIC_PATH)
    print(f'\nCoordinates loaded: {len(df_geo)} wells')
    print(df_geo.head().to_string(index=False))
else:
    print(f'\n{GEOGRAPHIC_PATH} not found.')
    print('   Export coordinates from QGIS and save to:')
    print(f'   {GEOGRAPHIC_PATH}')
    df_geo = pd.DataFrame()

# df_geographic: consolidate metrics + coordinates + directional data
df_geographic = df_metr.copy()

if not df_dir_summary.empty:
    df_geographic = df_geographic.merge(df_dir_summary, on='Well_Name', how='left')

if not df_geo.empty:
    df_geographic = df_geographic.merge(df_geo, on='Well_Name', how='left')

# Wells without directional survey → assume vertical
dir_cols_num = ['Max_INCL', 'MD_final', 'TVD_final', 'NS_final',
                'EW_final', 'Max_MD_TVD_diff', 'Pct_Deviated', 'Desl_H']
# Ensure Dir_Category exists even without directional data
if 'Dir_Category' not in df_geographic.columns:
    df_geographic['Dir_Category'] = 'Vertical (≤5°)'

dir_cols_cat = ['Dir_Category']

n_sem_survey = df_geographic[dir_cols_num].isnull().any(axis=1).sum()
if n_sem_survey > 0:
    df_geographic[dir_cols_num] = df_geographic[dir_cols_num].fillna(0)
    df_geographic[dir_cols_cat] = df_geographic[dir_cols_cat].fillna('Vertical (≤5°)')
    print(f'{n_sem_survey} well(s) without directional survey → assumed vertical')

print(f'\ndf_geographic: {len(df_geographic)} wells | columns: {list(df_geographic.columns)}')
print(df_geographic.to_string(index=False))

In [ ]:
# Merge: bring Lithology + Formation into the predictions DataFrame
cols_merge = [c for c in ['DEPTH', 'Well_Name', 'Lithology',
                           'Lithology_Confidence', 'Lithology_Segment_ID',
                           'Lithology_Segment_Thickness', 'Formation']
              if c in df_main.columns]

df = df_pred.merge(df_main[cols_merge], on=['DEPTH', 'Well_Name'], how='left')

# If Formation was not in the dataset, assign it via formations.csv
if 'Formation' not in df.columns or df['Formation'].isna().all():
    print('Assigning formations via formations.csv...')
    df['Formation'] = 'out_of_range'
    for well_name, group in df_form.groupby('Well_Name'):
        well_mask = df['Well_Name'] == well_name
        depths    = df.loc[well_mask, 'DEPTH'].values
        for _, row in group.iterrows():
            interval_mask = (depths >= row['depth_start']) & (depths <= row['depth_end'])
            df.loc[well_mask, 'Formation'] = np.where(
                interval_mask, row['formation'], df.loc[well_mask, 'Formation'].values
            )

# Residuals and errors
df['residual']     = df['DT_pred'] - df['DT_real']
df['abs_error']    = df['residual'].abs()
df['sq_error']     = df['residual'] ** 2
df['Lithology_norm'] = df['Lithology'].str.lower().str.strip()

# Merge global per-well metrics
df = df.merge(df_metr[['Well_Name', 'R2', 'RMSE', 'MAE']], on='Well_Name', how='left')

print(f'Final dataset: {len(df):,} samples')
print(f'Samples without lithology: {df["Lithology"].isna().sum():,}')
print(f'Samples without formation: {(df["Formation"]=="out_of_range").sum():,}')
print(f'\nDistribution by formation:')
print(df['Formation'].value_counts())
df.head(5)


## 3. Exploratory Overview — Well Performance (LOWO Ranking)

In [ ]:
# Dominant lithology per well
lith_dominant = (
    df.groupby('Well_Name')['Lithology_norm']
    .agg(lambda x: x.value_counts().index[0] if len(x.dropna()) > 0 else 'N/A')
    .reset_index()
    .rename(columns={'Lithology_norm': 'Lith_Dominant'})
)

well_stats = df_metr.merge(lith_dominant, on='Well_Name').sort_values('R2')
well_stats['R2_rank'] = range(1, len(well_stats) + 1)
print(f'well_stats created: {len(well_stats)} wells')


In [ ]:
THRESHOLD_RUIM = 0.70
THRESHOLD_BOM  = 0.90

worst = well_stats[well_stats['R2'] < THRESHOLD_RUIM]
best = well_stats[well_stats['R2'] >= THRESHOLD_BOM]

print(f'=== Wells with R² < {THRESHOLD_RUIM} ({len(worst)} wells) ===')
if worst.empty:
    print('No wells below the threshold.')
else:
    print(worst[['Well_Name', 'R2', 'RMSE', 'MAE', 'N_samples_test', 'Lith_Dominant']].to_string(index=False))

print(f'\n=== Wells with R² ≥ {THRESHOLD_BOM} ({len(best)} wells) ===')
if best.empty:
    print('No wells above the threshold.')
else:
    print(best[['Well_Name', 'R2', 'RMSE', 'MAE', 'Lith_Dominant']].to_string(index=False))

In [ ]:
# Chart: individual R² per well
def r2_bar_color(r2):
    if r2 < 0.70:  return '#E74C3C'
    return '#27AE60'

fig, ax = plt.subplots(figsize=(10, 10))

colors = [r2_bar_color(r2) for r2 in well_stats['R2']]
bars = ax.barh(well_stats['Well_Name'], well_stats['R2'],
               color=colors, edgecolor='white', height=0.7)

r2_mean = well_stats['R2'].mean()
ax.axvline(r2_mean, color='grey', alpha=0.7, linestyle='--', linewidth=1.2)
ax.axvline(0.70, color='#E74C3C', linestyle=':', linewidth=1.2)

for bar, val in zip(bars, well_stats['R2']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=7.5)

legend_patches = [
    mpatches.Patch(color='#E74C3C', label='R² < 0.70'),
    mpatches.Patch(color='#27AE60', label='R² ≥ 0.70'),
    plt.Line2D([0], [0], color='grey', alpha=0.7, linestyle='--', label=f'Average R² = {r2_mean:.4f}'),
    plt.Line2D([0], [0], color='#E74C3C', linestyle=':', label='Threshold R² = 0.70'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
ax.set_xlabel('R² (LOWO)', fontsize=12)
ax.set_title(f'{MODEL_NAME} — Individual R² per Well (LOWO)\n27 wells | Sergipe-Alagoas Basin', fontsize=13)
ax.set_xlim(0, 1.05)
ax.set_xticks([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ax.grid(axis='x', linestyle='--', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'xgboost_individual_r2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Wells with R² < THRESHOLD_RUIM — used across multiple sections
POOR_WELLS = [
    '3-BRSA-1069-SES',
    '1-BRSA-930D-SES',
    '3-BRSA-1207-SES',
    '1-BRSA-1013-SES',
]

In [ ]:
# Distribution by range
bins   = [-np.inf, 0.5, 0.70, 0.75, np.inf]
labels = ['< 0.50', '0.50–0.70', '0.70–0.75', '≥ 0.75']
well_stats['R2_category'] = pd.cut(well_stats['R2'], bins=bins, labels=labels)

print('Distribution of wells by R² range:')
print(well_stats['R2_category'].value_counts().sort_index())
print(f'\nR² min  : {well_stats["R2"].min():.4f} — {well_stats.loc[well_stats["R2"].idxmin(), "Well_Name"]}')
print(f'R² max  : {well_stats["R2"].max():.4f} — {well_stats.loc[well_stats["R2"].idxmax(), "Well_Name"]}')
print(f'R² median  : {well_stats["R2"].median():.4f}')
print(f'R² std     : {well_stats["R2"].std():.4f}')

In [ ]:
# Depth histogram + MAE by depth
fig, axes = plt.subplots(1, 2, figsize=(12, 10), sharey=True)

# Left panel: sample histogram by depth × lithology
DEPTH_BIN_HIST = 200  # 200m bins

df_hist = df.copy()
df_hist['DEPTH_bin'] = (df_hist['DEPTH'] // DEPTH_BIN_HIST * DEPTH_BIN_HIST).astype(int)

lith_order_hist = df['Lithology_norm'].value_counts().index.tolist()
pivot = (df_hist.groupby(['DEPTH_bin', 'Lithology_norm'])
         .size().unstack(fill_value=0))

for lith in lith_order_hist:
    if lith not in pivot.columns:
        pivot[lith] = 0
pivot = pivot[lith_order_hist]

depth_bins = pivot.index.values
bottom     = np.zeros(len(depth_bins))

for lith in lith_order_hist:
    vals = pivot[lith].values
    axes[0].barh(depth_bins, vals, height=DEPTH_BIN_HIST * 0.85,
                 left=bottom, color=lith_color(lith), alpha=0.85,
                 label=lith.replace('_', ' ').capitalize())
    bottom += vals

axes[0].invert_yaxis()
axes[0].set_xlabel('Number of samples', fontsize=11)
axes[0].set_ylabel('Depth [m]', fontsize=11)
axes[0].set_title('Sample distribution\nby depth and lithology', fontsize=11)
axes[0].legend(fontsize=8, loc='upper right', title='Lithology',
               framealpha=0.9, ncol=1)
axes[0].grid(alpha=0.3, axis='x')

# Right panel: mean MAE by depth bin
df_mae = df.copy()
df_mae['DEPTH_bin'] = (df_mae['DEPTH'] // DEPTH_BIN_HIST * DEPTH_BIN_HIST).astype(int)
mae_trend = (df_mae.groupby('DEPTH_bin')
             .agg(MAE_mean = ('abs_error', 'mean'),
                  MAE_std  = ('abs_error', 'std'),
                  N        = ('abs_error', 'count'))
             .reset_index()
             .query('N >= 50'))

axes[1].plot(mae_trend['MAE_mean'], mae_trend['DEPTH_bin'],
             color='steelblue', linewidth=2.5, marker='o', markersize=4,
             label='Mean MAE per bin')
axes[1].fill_betweenx(mae_trend['DEPTH_bin'],
                      mae_trend['MAE_mean'] - mae_trend['MAE_std'],
                      mae_trend['MAE_mean'] + mae_trend['MAE_std'],
                      alpha=0.2, color='steelblue', label='±1 std deviation')

axes[1].text(0.97, 0.03, 'Spearman ρ = -0.633  (p = 0.001)',
             transform=axes[1].transAxes, ha='right', va='bottom', fontsize=9,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

axes[1].set_xlabel('Mean MAE [µs/ft]', fontsize=11)
axes[1].set_title('Mean MAE by depth\n(trend ± standard deviation)', fontsize=11)
axes[1].legend(fontsize=9, loc='upper right')
axes[1].grid(alpha=0.3)

plt.suptitle(f'{MODEL_NAME} — Data Distribution and Error by Depth\n'
             f'Sergipe-Alagoas Basin | {df["Well_Name"].nunique()} wells',
             fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'histogram_depth_mae.png',
            dpi=150, bbox_inches='tight')
plt.show()
print(f'{DEPTH_BIN_HIST}m bins | Total: {len(df):,} samples')

In [ ]:
# General overview: performance by lithology (all formations)
summary = (
    df.groupby('Lithology')
    .agg(
        N=('abs_error', 'count'),
        MAE=('abs_error', 'mean'),
        RMSE=('residual', lambda x: (x**2).mean()**0.5),
        Bias=('residual', 'mean'),
    )
    .reset_index()
    .sort_values('N', ascending=False)
    .round(2)
)

# Percentage of total dataset
summary['N_pct'] = (summary['N'] / summary['N'].sum() * 100).round(1)

print(f"Total samples: {summary['N'].sum():,}")
print()
print(summary[['Lithology', 'N', 'N_pct', 'MAE', 'RMSE', 'Bias']].to_string(index=False))

In [ ]:
# Table: performance by lithology
summary_display = summary[['Lithology', 'N', 'N_pct', 'MAE', 'RMSE', 'Bias']].copy()
summary_display.columns = ['Lithology', 'N samples', 'N (%)', 'MAE (µs/ft)', 'RMSE (µs/ft)', 'Bias (µs/ft)']

print(summary_display.to_string(index=False))

In [ ]:
# Chart: sample count and MAE by lithology (2 decimal places in %)
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle('XGBoost — LOWO Performance by Lithology\nSergipe-Alagoas Basin | 27 wells | 231,815 samples',
             fontsize=13, fontweight='bold')

liths  = summary['Lithology'].tolist()
n_vals = summary['N'].tolist()
mae    = summary['MAE'].tolist()
bias   = summary['Bias'].tolist()
n_pct  = (summary['N'] / summary['N'].sum() * 100).tolist()
y      = range(len(liths))

# Panel 1: N samples
ax1 = axes[0]
bars1 = ax1.barh(y, n_vals, color='steelblue', alpha=0.75, edgecolor='white')
ax1.set_xscale('log')
ax1.set_xlabel('Number of samples (log scale)', fontsize=11)
ax1.set_title('Sample count', fontsize=12, fontweight='bold')
ax1.set_yticks(y)
ax1.set_yticklabels(liths, fontsize=10)
ax1.invert_yaxis()

for bar, n, pct in zip(bars1, n_vals, n_pct):
    ax1.text(bar.get_width() * 1.05, bar.get_y() + bar.get_height()/2,
             f'{n:,} ({pct:.2f}%)', va='center', fontsize=8.5)

ax1.set_xlim(right=ax1.get_xlim()[1] * 8)

# Panel 2: MAE colored by bias
ax2 = axes[1]
bar_colors = ['#d73027' if b > 0 else '#4575b4' for b in bias]
bars2 = ax2.barh(y, mae, color=bar_colors, alpha=0.80, edgecolor='white')
ax2.set_xlabel('Mean Absolute Error [µs/ft]', fontsize=11)
ax2.set_title('MAE (colored by bias direction)', fontsize=12, fontweight='bold')
ax2.axvline(x=sum(mae)/len(mae), color='gray', linestyle='--',
           linewidth=1, label=f'Mean MAE = {sum(mae)/len(mae):.2f} µs/ft\n(unweighted)')

for bar, m, b in zip(bars2, mae, bias):
    ax2.text(bar.get_width() + 0.05,
             bar.get_y() + bar.get_height()/2,
             f'{m:.2f}  (bias {b:+.2f})',
             va='center', fontsize=8.5)

ax2.set_xlim(right=ax2.get_xlim()[1] * 1.6)
patch_red  = mpatches.Patch(color='#d73027', alpha=0.8, label='Overestimates DT (bias > 0)')
patch_blue = mpatches.Patch(color='#4575b4', alpha=0.8, label='Underestimates DT (bias < 0)')
mean_line = Line2D([], [], color='gray', linestyle='--', linewidth=1,
                   label=f'Mean MAE = {sum(mae)/len(mae):.2f} µs/ft (unweighted)')
ax2.legend(handles=[patch_red, patch_blue, mean_line], fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'lithology_performance.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Lithology metrics — ALL lithologies present
lith_metrics = []
for lith, grp in df.dropna(subset=['Lithology_norm', 'DT_real', 'DT_pred']).groupby('Lithology_norm'):
    n_samples = len(grp)
    n_wells   = grp['Well_Name'].nunique()

    # R² requires at least 2 samples
    if n_samples < 2:
        continue

    lith_metrics.append({
        'Lithology'        : lith,
        'R2'               : r2_score(grp['DT_real'], grp['DT_pred']),
        'RMSE'             : np.sqrt(mean_squared_error(grp['DT_real'], grp['DT_pred'])),
        'MAE'              : mean_absolute_error(grp['DT_real'], grp['DT_pred']),
        'Bias'             : grp['residual'].mean(),
        'N_samples'        : n_samples,
        'N_wells'          : n_wells,
        'Representatividade'   : representativity_label(n_samples, n_wells),
    })

df_lith = pd.DataFrame(lith_metrics).sort_values('R2', ascending=False)

# Separate lithologies with high/moderate representativity for main analyses
valid_liths_all  = df_lith['Lithology'].tolist()
valid_liths_main = df_lith[
    df_lith['Representatividade'].str.contains('High|Moderate')
]['Lithology'].tolist()

print('Performance by Lithology (all)\n')
cols = ['Lithology', 'R2', 'RMSE', 'MAE', 'Bias', 'N_samples', 'N_wells', 'Representatividade']
print(df_lith[cols].to_string(index=False, float_format='{:.4f}'.format))

## 4. Hypothesis H1 — Depth

> **Hypothesis:** Prediction error increases with depth due to increasing compaction,
> cementation, and diagenesis, which alter DT in ways not fully captured by the input features.

> **Method:** Bin samples into 200 m depth intervals; compute mean MAE and bias per bin;
> assess statistical correlation (Spearman ρ) between depth and MAE; check for confounding
> via sample count per bin (partial correlation controlling for N).

In [ ]:
# MAE and bias by depth bin (all wells)
DEPTH_BIN_SIZE = 200   # meters

df['DEPTH_bin'] = (df['DEPTH'] // DEPTH_BIN_SIZE * DEPTH_BIN_SIZE).astype(int)

depth_stats = (
    df.groupby('DEPTH_bin')
    .agg(MAE_mean     =('abs_error', 'mean'),
         residual_mean=('residual',  'mean'),
         N            =('residual',  'count'),
         DT_real_mean =('DT_real',   'mean'),
         DT_pred_mean =('DT_pred',   'mean'))
    .reset_index()
    .query('N >= 50')
)

fig, axes = plt.subplots(1, 3, figsize=(16, 8))
kw = dict(linewidth=1.8, marker='o', markersize=3)

axes[0].plot(depth_stats['MAE_mean'],      depth_stats['DEPTH_bin'], color='#E74C3C', **kw)
axes[0].invert_yaxis()
axes[0].set_ylabel('Depth [m]', fontsize=11)
axes[0].set_xlabel('Mean MAE [µs/ft]', fontsize=11)
axes[0].set_title('Absolute Error\nvs Depth')

axes[1].plot(depth_stats['residual_mean'], depth_stats['DEPTH_bin'], color='#2980B9', **kw)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].invert_yaxis()
axes[1].set_xlabel('Mean residual [µs/ft]', fontsize=11)
axes[1].set_title('Bias (DT_pred − DT_real)\nvs Depth')
axes[1].set_yticklabels([])

axes[2].plot(depth_stats['DT_real_mean'], depth_stats['DEPTH_bin'],
             color='black', linewidth=1.5, label='DT real')
axes[2].plot(depth_stats['DT_pred_mean'], depth_stats['DEPTH_bin'],
             color='#E67E22', linewidth=1.5, linestyle='--', label='DT pred')
axes[2].invert_yaxis()
axes[2].set_xlabel('DT médio [µs/ft]', fontsize=11)
axes[2].set_title('Real vs Predicted DT\nby Depth')
axes[2].legend(fontsize=9)
axes[2].set_yticklabels([])

for ax in axes:
    ax.grid(alpha=0.3)

plt.suptitle(f'{MODEL_NAME} — Errors vs Depth\n{DEPTH_BIN_SIZE}m bins | All wells aggregated', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'errors_depth.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: Depth vs MAE — separated by lithology
df_scatter = df[df['Lithology_norm'].isin(valid_liths_main)].copy()

n_liths = len(valid_liths_main)
n_cols  = 4
n_rows  = (n_liths + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 7 * n_rows))
axes_flat = axes.flatten()

for i, (ax, lith) in enumerate(zip(axes_flat, valid_liths_main)):
    sub = df_scatter[df_scatter['Lithology_norm'] == lith].copy()

    # Subsample if needed
    if len(sub) > 5000:
        sub = sub.sample(5000, random_state=42)

    ax.scatter(sub['abs_error'], sub['DEPTH'],
               color=lith_color(lith), alpha=0.25, s=6)

    # Trend line per depth bin
    sub['DEPTH_bin'] = (sub['DEPTH'] // DEPTH_BIN_SIZE * DEPTH_BIN_SIZE).astype(int)
    trend = (sub.groupby('DEPTH_bin')
               .agg(MAE_mean=('abs_error', 'mean'), N=('abs_error', 'count'))
               .reset_index()
               .query('N >= 10'))
    if len(trend) >= 2:
        ax.plot(trend['MAE_mean'], trend['DEPTH_bin'],
                color='black', linewidth=2, linestyle='--', zorder=5)

    ax.invert_yaxis()
    n_total = df_scatter[df_scatter['Lithology_norm'] == lith].shape[0]
    ax.set_title(f'{lith.replace("_", " ").capitalize()}\n(N={n_total:,})', fontsize=10)
    ax.set_xlabel('MAE [µs/ft]', fontsize=9)
    ax.grid(alpha=0.3)
    if i % n_cols == 0:
        ax.set_ylabel('Depth [m]', fontsize=10)

# Hide empty subplots
for ax in axes_flat[n_liths:]:
    ax.set_visible(False)

plt.suptitle(f'{MODEL_NAME} — Absolute Error vs Depth by Lithology\n'
             f'(dashed line = mean trend | High/Moderate representativity only)',
             fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'scatter_error_depth_lithology.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical correlation: depth × MAE
rho, p_spearman = spearmanr(depth_stats['DEPTH_bin'], depth_stats['MAE_mean'])
r,   p_pearson  = pearsonr( depth_stats['DEPTH_bin'], depth_stats['MAE_mean'])

print('=== Depth × MAE Correlation (all wells) ===')
print(f'Spearman ρ = {rho:+.3f}  (p = {p_spearman:.4f})')
print(f'Pearson  r = {r:+.3f}  (p = {p_pearson:.4f})')

if abs(rho) < 0.2:
    print('→ Weak correlation: depth is not the dominant error factor')
elif rho > 0:
    print('→ Positive correlation: errors INCREASE with depth')
else:
    print('→ Negative correlation: errors DECREASE with depth (shallow zones are harder)')

In [ ]:
# Confounding check: N samples per depth bin × MAE
# Tests whether high MAE in shallow zones is explained by
# low sample count (representativity) or is a genuine geological effect.

# depth_stats already has N and MAE_mean
rho_n, p_n = spearmanr(depth_stats["N"], depth_stats["MAE_mean"])
rho_d, p_d = spearmanr(depth_stats["DEPTH_bin"], depth_stats["N"])

print("=== Confounding check: N samples × MAE ===")
print(f"Spearman ρ (N × MAE)     = {rho_n:+.3f}  (p = {p_n:.4f})")
print(f"Spearman ρ (DEPTH × N)   = {rho_d:+.3f}  (p = {p_d:.4f})")
print()

# Partial correlation (depth × MAE, controlling for N)
def residualize(y, x):
    slope = np.cov(x, y)[0, 1] / np.var(x)
    intercept = np.mean(y) - slope * np.mean(x)
    return y - (slope * x + intercept)

mae_resid   = residualize(depth_stats["MAE_mean"].values, depth_stats["N"].values)
depth_resid = residualize(depth_stats["DEPTH_bin"].values, depth_stats["N"].values)

rho_partial, p_partial = spearmanr(depth_resid, mae_resid)

print(f"Partial Spearman ρ (DEPTH × MAE | controlling for N) = {rho_partial:+.3f}  (p = {p_partial:.4f})")
print()

if abs(rho_n) > 0.4 and p_n < 0.05:
    print("→ N samples correlates with MAE: sample count may confound the depth effect.")
    print(f"  After controlling for N, residual depth effect: ρ = {rho_partial:+.3f} (p = {p_partial:.4f})")
    if abs(rho_partial) > 0.3 and p_partial < 0.05:
        print("  → Depth effect persists after controlling for N: genuinely geological.")
    else:
        print("  → Depth effect weakens after controlling for N: partially explained by representativity.")
else:
    print("→ N samples does not significantly correlate with MAE.")
    print("  → The depth effect is unlikely to be explained by sample count alone.")

# Figure: dual depth-profile panel (N × MAE)
fig, (ax_n, ax_mae) = plt.subplots(1, 2, figsize=(10, 8), sharey=True)
fig.subplots_adjust(wspace=0.08)

depth_vals = depth_stats["DEPTH_bin"].values
n_vals     = depth_stats["N"].values
mae_vals   = depth_stats["MAE_mean"].values

# Left panel — N samples (horizontal bars)
ax_n.barh(depth_vals, n_vals, height=150, color="steelblue", alpha=0.75,
          edgecolor="white", linewidth=0.5)
ax_n.set_xlabel("N samples per 200 m bin", fontsize=11)
ax_n.set_ylabel("Depth [m]", fontsize=11)
ax_n.set_title(f"Sample count ρ(DEPTH × N) = {rho_d:+.3f}  (p = {p_d:.3f})",
               fontsize=10)
ax_n.invert_yaxis()

# Right panel — mean MAE line
ax_mae.plot(mae_vals, depth_vals,
            color="tomato", linewidth=2, marker="o", markersize=5,
            markerfacecolor="white", markeredgewidth=1.5)
ax_mae.set_xlabel("Mean MAE [µs/ft]", fontsize=11)
ax_mae.set_title(f"Prediction error ρ(N × MAE) = {rho_n:+.3f} \n (p = {p_n:.3f}) | "\
    f"partial ρ = {rho_partial:+.3f}  (p = {p_partial:.3f})",
    fontsize=10)
ax_mae.yaxis.set_tick_params(labelleft=False)

# Visual reference: mean MAE line
mean_mae = mae_vals.mean()
ax_mae.axvline(mean_mae, color="gray", linestyle="--", linewidth=1, alpha=0.6)
ax_mae.text(mean_mae + 0.3, depth_vals.max() * 0.97,
            f"mean = {mean_mae:.1f}", color="gray", fontsize=8)

plt.suptitle(f"H1 — Confounding check: does sample count explain the depth effect? \n "\
    f"ρ(DEPTH × MAE) = {rho:+.3f}  (p = {p_spearman:.4f})",
    fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "confounding_n_mae.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Summary:")
print(f"  ρ(DEPTH × MAE)             = {rho:+.3f}  (p = {p_spearman:.4f})")
print(f"  ρ(N × MAE)                 = {rho_n:+.3f}  (p = {p_n:.4f})")
print(f"  ρ(DEPTH × MAE | N)         = {rho_partial:+.3f}  (p = {p_partial:.4f})")

## 5. Hypothesis H2 — DT Response Extrapolation

> **Hypothesis:** Poorly performing wells exhibit DT values outside the P5–P95 range of the
> LOWO training set, causing the model to extrapolate beyond its calibrated response domain.

> **Method:** (1) Overlay DT distributions of each poor-performing well against the training
> set histogram, marking the P5–P95 bounds; compute the percentage of test samples outside
> this interval. (2) Plot a calibration curve (absolute error × real DT) to visualize whether
> error increases at the extremes of the DT range.

### 5.1 DT Distribution — Poor-Performing Wells vs Training Set

Verifies whether the real DT of the poor-performing wells falls within the P5–P95 interval of the LOWO training set. Grounds the taxonomy: *response extrapolation* vs *feature extrapolation*.

In [ ]:
# Real DT distribution: poor-performing wells vs training set
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes_flat = axes.flatten()

for ax, well in zip(axes_flat, POOR_WELLS):
    r2 = df_metr.loc[df_metr['Well_Name']==well, 'R2'].values[0]

    # Training set DT (all other wells)
    dt_train = df[df['Well_Name'] != well]['DT_real']

    # Poor-performing well DT
    dt_poor  = df[df['Well_Name'] == well]['DT_real']

    # Common limits
    x_min = min(dt_train.min(), dt_poor.min()) - 2
    x_max = max(dt_train.max(), dt_poor.max()) + 2

    # Training histogram
    ax.hist(dt_train, bins=80, range=(x_min, x_max),
            color='steelblue', alpha=0.5, density=True,
            label=f'Training (N={len(dt_train):,})\nMean={dt_train.mean():.1f} | Std={dt_train.std():.1f}')

    # Poor-performing well histogram
    ax.hist(dt_poor, bins=40, range=(x_min, x_max),
            color='crimson', alpha=0.7, density=True,
            label=f'{well.replace("-SES","")}\n(N={len(dt_poor):,})\nMean={dt_poor.mean():.1f} | Std={dt_poor.std():.1f}')

    # P5 and P95 percentiles of training set
    p5_train  = dt_train.quantile(0.05)
    p95_train = dt_train.quantile(0.95)
    ax.axvline(p5_train,  color='steelblue', lw=1.5, linestyle='--',
               alpha=0.8, label=f'P5 training = {p5_train:.1f}')
    ax.axvline(p95_train, color='steelblue', lw=1.5, linestyle=':',
               alpha=0.8, label=f'P95 training = {p95_train:.1f}')

    # % of well samples outside P5-P95 range of training set
    pct_fora = ((dt_poor < p5_train) | (dt_poor > p95_train)).mean() * 100
    ax.text(0.97, 0.97,
            f'R² = {r2:.3f}\n'
            f'Outside P5–P95 training: {pct_fora:.1f}%',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, bbox=dict(facecolor='white', alpha=0.85,
                                  edgecolor='gray', pad=3))

    ax.set_xlabel('Real DT [µs/ft]', fontsize=10, fontweight='bold')
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(f'{well}\nR²={r2:.3f}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7.5, loc='upper left', framealpha=0.85)
    ax.grid(alpha=0.25)

fig.suptitle(f'{MODEL_NAME} — Real DT Distribution: Poor-Performing Wells vs Training Set\n'
             f'Blue = training (27 wells) | Red = test well | lines = P5 and P95 of training',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'real_dt_distribution_vs_poor.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Numerical summary table
print('DT Distribution: Poor-Performing Wells vs Training Set\n')
print(f'{"Well":25s} {"R²":>6s} {"DT_mean":>8s} {"DT_std":>7s} '
      f'{"DT_min":>7s} {"DT_max":>7s} {"P5_training":>10s} {"P95_training":>10s} {"% out P5-P95":>14s}')
print('-' * 110)

for well in POOR_WELLS:
    r2       = df_metr.loc[df_metr['Well_Name']==well, 'R2'].values[0]
    dt_train = df[df['Well_Name'] != well]['DT_real']
    dt_poor  = df[df['Well_Name'] == well]['DT_real']
    p5       = dt_train.quantile(0.05)
    p95      = dt_train.quantile(0.95)
    pct_fora = ((dt_poor < p5) | (dt_poor > p95)).mean() * 100

    print(f'{well:25s} {r2:6.3f} {dt_poor.mean():8.1f} {dt_poor.std():7.1f} '
          f'{dt_poor.min():7.1f} {dt_poor.max():7.1f} {p5:10.1f} {p95:10.1f} {pct_fora:13.1f}%')

### 5.2 Calibration Curve — Error vs Real DT

Assesses whether the model errs systematically in certain DT ranges, revealing directional bias by petrophysical regime.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes_flat = axes.flatten()

print('\n Analysis 2: Error vs real DT — model calibration\n')
print(f'{"DT Range":15s}', end='')
for well in POOR_WELLS:
    print(f' {well.replace("-SES",""):>22s}', end='')
print()
print('-' * 110)

# DT bins for table
dt_bins   = [50, 70, 80, 90, 100, 110, 120, 130, 160]
dt_labels = [f'{dt_bins[i]}–{dt_bins[i+1]}' for i in range(len(dt_bins)-1)]

rows_table = {lbl: [] for lbl in dt_labels}

for ax, well in zip(axes_flat, POOR_WELLS):
    r2       = df_metr.loc[df_metr['Well_Name']==well, 'R2'].values[0]
    df_poor  = df[df['Well_Name'] == well].copy()
    dt_train = df[df['Well_Name'] != well]['DT_real']
    p5_train = dt_train.quantile(0.05)
    p95_train= dt_train.quantile(0.95)
    
    # Scatter (subsample)
    sub = df_poor.sample(min(len(df_poor), 3000), random_state=42)
    ax.scatter(sub['DT_real'], sub['abs_error'],
               c=sub['abs_error'], cmap='RdYlGn_r',
               vmin=0, vmax=15, s=8, alpha=0.4, zorder=2)
    
    # Average trend per DT bin
    df_poor['DT_bin'] = pd.cut(df_poor['DT_real'], bins=dt_bins, labels=dt_labels)
    trend = (df_poor.groupby('DT_bin', observed=True)['abs_error']
             .agg(['mean', 'count'])
             .reset_index()
             .query('count >= 10'))
    
    # Pegar ponto médio de cada bin para plotar
    bin_mids = [(dt_bins[i] + dt_bins[i+1]) / 2
                for i in range(len(dt_bins)-1)
                if dt_labels[i] in trend['DT_bin'].values]
    
    ax.plot([b for b, l in zip(
                [(dt_bins[i]+dt_bins[i+1])/2 for i in range(len(dt_bins)-1)],
                dt_labels) if l in trend['DT_bin'].values],
            trend['mean'].values,
            color='black', lw=2, linestyle='--', zorder=5,
            label='Mean MAE per bin')
    
    # Zona de extrapolação
    ax.axvspan(p95_train, df_poor['DT_real'].max() + 2,
               color='#E74C3C', alpha=0.08, label=f'Above P95 training ({p95_train:.0f})')
    ax.axvspan(df_poor['DT_real'].min() - 2, p5_train,
               color='#2980B9', alpha=0.08, label=f'Below P5 training ({p5_train:.0f})')
    
    ax.axvline(p5_train,  color='steelblue', lw=1.2, linestyle='--', alpha=0.7)
    ax.axvline(p95_train, color='red',       lw=1.2, linestyle='--', alpha=0.7)
    
    ax.set_xlabel('Real DT [µs/ft]', fontsize=10, fontweight='bold')
    ax.set_ylabel('Absolute Error [µs/ft]', fontsize=10)
    ax.set_title(f'{well}\nR²={r2:.3f}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7, loc='upper left', framealpha=0.85)
    ax.grid(alpha=0.25)
    ax.set_ylim(0, None)
    
    # Numerical table
    for lbl in dt_labels:
        row = trend[trend['DT_bin'] == lbl]
        rows_table[lbl].append(f'{row["mean"].values[0]:.2f} (N={row["count"].values[0]:,})'
                                if len(row) > 0 else '—')

for lbl in dt_labels:
    print(f'{lbl:15s}', end='')
    for val in rows_table[lbl]:
        print(f' {val:>22s}', end='')
    print()

fig.suptitle(f'{MODEL_NAME} — Absolute Error vs Real DT (Calibration Curve)\n'
             f'Red zone = above P95 training | Blue zone = below P5 training',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'calibration_curve.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 6. Hypothesis H3 — Feature Space Extrapolation

> **Hypothesis:** Poorly performing wells have input features (GR, RT90, RHOB, NPHI) outside
> the P5–P95 range of the training set, placing them in an extrapolation regime in feature space.

> **Method:** (1) For each feature, compare the distribution of the poor-performing well against
> the training set; quantify the percentage of samples outside P5–P95. (2) Identify the nearest
> neighbor in the normalized GR×RHOB×NPHI space (KNN k=1) to assess whether the training set
> contains geologically similar samples.

### 6.1 Feature Space Comparison

For each poor-performing well, computes the percentage of samples whose features (GR, RT90, RHOB, NPHI) fall outside the P5–P95 of the corresponding training set.

In [ ]:
# MAE by lithology in poor-performing wells
print('MAE by Lithology — Poor-Performing Wells\n')

for well in POOR_WELLS:
    sub = df[df['Well_Name'] == well]
    r2  = df_metr.loc[df_metr['Well_Name']==well, 'R2'].values[0]
    
    lith_stats = (
        sub.groupby('Lithology_norm')
        .agg(
            N       = ('abs_error', 'count'),
            MAE     = ('abs_error', 'mean'),
            Bias    = ('residual',  'mean'),
            P50_err = ('abs_error', lambda x: x.quantile(0.50)),
            P90_err = ('abs_error', lambda x: x.quantile(0.90)),
        )
        .reset_index()
        .query('N >= 10')
        .sort_values('MAE', ascending=False)
    )
    
    print(f'{well} (R²={r2:.3f}):')
    print(lith_stats.to_string(index=False, float_format='{:.2f}'.format))
    print()

In [ ]:
features = ['GR', 'RT90', 'RHOB', 'NPHI']
feat_labels = {'GR': 'GR [API]', 'RT90': 'RT90 [ohm.m]',
               'RHOB': 'RHOB [g/cm³]', 'NPHI': 'NPHI [pu]'}

fig, axes = plt.subplots(len(POOR_WELLS), len(features),
                          figsize=(18, 14))

print('Features outside P5–P95 of training set\n')
print(f'{"Well":25s} {"R²":>6s}', end='')
for f in features:
    print(f' {f:>12s}', end='')
print()
print('-' * 100)

for row_i, well in enumerate(POOR_WELLS):
    r2       = df_metr.loc[df_metr['Well_Name']==well, 'R2'].values[0]
    df_train = df[df['Well_Name'] != well]
    df_poor  = df[df['Well_Name'] == well]
    
    pct_fora_list = []
    
    for col_j, feat in enumerate(features):
        ax = axes[row_i, col_j]
        
        train_vals = df_train[feat].dropna()
        poor_vals  = df_poor[feat].dropna()
        
        p5  = train_vals.quantile(0.05)
        p95 = train_vals.quantile(0.95)
        pct_fora = ((poor_vals < p5) | (poor_vals > p95)).mean() * 100
        pct_fora_list.append(f'{pct_fora:.1f}%')
        
        x_min = min(train_vals.quantile(0.01), poor_vals.quantile(0.01)) * 0.95
        x_max = max(train_vals.quantile(0.99), poor_vals.quantile(0.99)) * 1.05
        
        ax.hist(train_vals, bins=60, range=(x_min, x_max),
                color='steelblue', alpha=0.5, density=True, label='Training')
        ax.hist(poor_vals, bins=40, range=(x_min, x_max),
                color='crimson', alpha=0.7, density=True,
                label=well.replace('-SES',''))
        ax.axvline(p5,  color='steelblue', lw=1.5, linestyle='--', alpha=0.8)
        ax.axvline(p95, color='steelblue', lw=1.5, linestyle=':', alpha=0.8)
        
        ax.text(0.97, 0.97, f'{pct_fora:.1f}% outside',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=8, color='red' if pct_fora > 10 else 'black',
                fontweight='bold' if pct_fora > 10 else 'normal',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))
        
        ax.set_xlabel(feat_labels[feat], fontsize=8)
        ax.grid(alpha=0.25)
        if feat == 'RT90':
            x_min = max(x_min, 1e-2)
            ax.set_xscale('log')

        if col_j == 0:
            ax.set_ylabel(f'{well.replace("-SES","")}\nR²={r2:.3f}\nDensidade',
                          fontsize=7.5)
        if row_i == 0:
            ax.set_title(feat, fontsize=10, fontweight='bold')
    
    print(f'{well:25s} {r2:6.3f}', end='')
    for p in pct_fora_list:
        print(f' {p:>12s}', end='')
    print()

fig.suptitle(f'{MODEL_NAME} — Features outside P5–P95 of Training Set\n'
             f'Blue = training (27 wells) | Red = test well | lines = P5/P95 training',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'features_outside_range_training.png',
            dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Nearest Neighbor in Feature Space (KNN k=1)

For each sample from the poor-performing wells, identifies the nearest neighbor in the normalized GR×RHOB×NPHI feature space of the training set. The gap (real DT − neighbor DT) indicates whether the model tends to under- or overestimate DT for that petrophysical context.

In [ ]:
print('\nNearest neighbor in feature space')
print('For each sample from the poor-performing well, which training well is most similar?\n')

feat_knn = ['GR', 'RHOB', 'NPHI']   # RT90 excluded — very variable and skewed

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes_flat = axes.flatten()

for ax, well in zip(axes_flat, POOR_WELLS):
    r2       = df_metr.loc[df_metr['Well_Name']==well, 'R2'].values[0]
    df_train = df[df['Well_Name'] != well].dropna(subset=feat_knn + ['DT_real']).copy()
    df_poor  = df[df['Well_Name'] == well].dropna(subset=feat_knn + ['DT_real']).copy()
    
    # Features normalization
    scaler = StandardScaler()
    X_train = scaler.fit_transform(df_train[feat_knn])
    X_poor  = scaler.transform(df_poor[feat_knn])
    
    # KNN k=1: nearest neighbor in training set for each sample of the well
    knn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree', n_jobs=-1)
    knn.fit(X_train)
    distances, indices = knn.kneighbors(X_poor)
    
    # Nearest neighbor well
    df_poor = df_poor.reset_index(drop=True)
    df_poor['nn_well'] = df_train.reset_index(drop=True).iloc[indices.flatten()]['Well_Name'].values
    df_poor['nn_dt']   = df_train.reset_index(drop=True).iloc[indices.flatten()]['DT_real'].values
    df_poor['nn_dist'] = distances.flatten()
    df_poor['dt_gap']  = df_poor['DT_real'] - df_poor['nn_dt']
    
    # Neighbor distribution per well
    nn_counts = df_poor['nn_well'].value_counts()
    
    print(f'\n{well} (R²={r2:.3f}):')
    print(f'  Mean measured DT     : {df_poor["DT_real"].mean():.1f} µs/ft')
    print(f'  Mean neighbor DT     : {df_poor["nn_dt"].mean():.1f} µs/ft')
    print(f'  Mean gap (measured−nbr)  : {df_poor["dt_gap"].mean():+.1f} µs/ft')
    print(f'  P90 gap              : {df_poor["dt_gap"].quantile(0.90):+.1f} µs/ft')
    print(f'  Most common neighbor well: {nn_counts.index[0]} ({nn_counts.iloc[0]/len(df_poor)*100:.1f}%)')
    print(f'  Top-3 neighbors:')
    for w, c in nn_counts.head(3).items():
        r2_nn = df_metr.loc[df_metr['Well_Name']==w, 'R2'].values[0]
        print(f'    {w} (R²={r2_nn:.3f}): {c:,} samples ({c/len(df_poor)*100:.1f}%)')
    
    # Chart: Real DT vs neighbor DT (scatter colored by gap)
    sub = df_poor.sample(min(len(df_poor), 3000), random_state=42)
    sc = ax.scatter(sub['nn_dt'], sub['DT_real'],
                    c=sub['abs_error'], cmap='RdYlGn_r',
                    vmin=0, vmax=15, s=8, alpha=0.5)
    
    # Line 1:1
    lims = [min(sub['nn_dt'].min(), sub['DT_real'].min()) - 2,
            max(sub['nn_dt'].max(), sub['DT_real'].max()) + 2]
    ax.plot(lims, lims, 'k--', lw=1.5, label='1:1 (real DT = neighbor DT)')
    
    plt.colorbar(sc, ax=ax, label='Abs. Error [µs/ft]', fraction=0.046, pad=0.04)
    
    gap_mean = df_poor['dt_gap'].mean()
    ax.text(0.03, 0.97,
            f'Mean gap: {gap_mean:+.1f} µs/ft\n'
            f'Neighbor: {nn_counts.index[0].replace("-SES","")}\n({nn_counts.iloc[0]/len(df_poor)*100:.0f}% of samples)',
            transform=ax.transAxes, ha='left', va='top', fontsize=8,
            bbox=dict(facecolor='white', alpha=0.85, edgecolor='gray', pad=3))
    
    ax.set_xlabel('Nearest neighbor DT [µs/ft]', fontsize=9)
    ax.set_ylabel('Well measured DT [µs/ft]', fontsize=9)
    ax.set_title(f'{well}\nR²={r2:.3f}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7.5, loc='lower right', framealpha=0.85)
    ax.grid(alpha=0.25)

fig.suptitle(f'{MODEL_NAME} — Measured DT vs Nearest Neighbor DT (GR × RHOB × NPHI)\n'
             f'If gap > 0: measured DT greater than neighbor → model tends to underestimate DT',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'knn_nearest_neighbor.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 7. Hypothesis H4 — Geological Formation

> **Hypothesis:** Wells in distinct depositional environments (deeper formations with higher
> compaction, cementation, and diagenetic alteration) show systematically different prediction
> performance, indicating that formation-scale petrophysical variability is not fully captured
> by the four input features (GR, RT90, RHOB, NPHI).

> **Method:** Compute per-formation metrics (R², RMSE, MAE, Bias); analyze formation composition
> of poor-performing wells; residual distribution by formation; geographic map of deepest
> formation per well; MAE heatmap and bubble chart by formation × lithology.

### 7.1 Performance by Formation

In [ ]:
# Formation metrics
df_valid = df[df['Formation'] != 'out_of_range'].copy()

form_metrics = []
for form, grp in df_valid.groupby('Formation'):
    n_samples = len(grp)
    n_wells   = grp['Well_Name'].nunique()
    if n_samples < 2:
        continue
    form_metrics.append({
        'Formation'         : form,
        'R2'                : r2_score(grp['DT_real'], grp['DT_pred']),
        'RMSE'              : np.sqrt(mean_squared_error(grp['DT_real'], grp['DT_pred'])),
        'MAE'               : mean_absolute_error(grp['DT_real'], grp['DT_pred']),
        'Bias'              : grp['residual'].mean(),
        'N_samples'         : n_samples,
        'N_wells'           : n_wells,
        'DT_mean'           : grp['DT_real'].mean(),
        'DT_std'            : grp['DT_real'].std(),
        'Representatividade': representativity_label(n_samples, n_wells),
    })

df_form_metrics = pd.DataFrame(form_metrics).sort_values('R2', ascending=False)

print('Performance by Formation (all)\n')
cols = ['Formation', 'R2', 'RMSE', 'MAE', 'Bias', 'N_samples', 'N_wells',
        'DT_mean', 'DT_std', 'Representatividade']
print(df_form_metrics[cols].to_string(index=False, float_format='{:.3f}'.format))


In [ ]:
# Chart: R², RMSE and Bias by formation
form_order  = df_form_metrics.sort_values('R2')['Formation'].tolist()
colors_form = [FORMATION_COLORS.get(f, '#BDC3C7') for f in form_order]
r2_global   = r2_score(df_valid['DT_real'], df_valid['DT_pred'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# R²
r2_vals = [df_form_metrics.loc[df_form_metrics['Formation']==f, 'R2'].values[0] for f in form_order]
bars = axes[0].barh(form_order, r2_vals, color=colors_form, edgecolor='white')
for bar, val in zip(bars, r2_vals):
    axes[0].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)
axes[0].axvline(r2_global, color='black', linestyle='--', lw=1.2,
                label=f'R² global = {r2_global:.3f}')
axes[0].set_xlabel('R²'); axes[0].set_title('R² by Formation')
axes[0].set_xlim(0, 1.1); axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3, axis='x')

# RMSE
rmse_vals = [df_form_metrics.loc[df_form_metrics['Formation']==f, 'RMSE'].values[0] for f in form_order]
bars = axes[1].barh(form_order, rmse_vals, color=colors_form, edgecolor='white')
for bar, val in zip(bars, rmse_vals):
    axes[1].text(val + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=9)
axes[1].set_xlabel('RMSE [µs/ft]'); axes[1].set_title('RMSE by Formation')
axes[1].grid(alpha=0.3, axis='x')

# Bias
bias_vals = [df_form_metrics.loc[df_form_metrics['Formation']==f, 'Bias'].values[0] for f in form_order]
bar_colors_bias = ['#E74C3C' if v > 0 else '#2980B9' for v in bias_vals]
bars = axes[2].barh(form_order, bias_vals, color=bar_colors_bias, edgecolor='white')
for bar, val in zip(bars, bias_vals):
    offset = 0.05 if val >= 0 else -0.05
    ha = 'left' if val >= 0 else 'right'
    axes[2].text(val + offset, bar.get_y() + bar.get_height()/2,
                 f'{val:+.2f}', va='center', ha=ha, fontsize=9)
axes[2].axvline(0, color='black', lw=1.0)
axes[2].set_xlabel('Bias [µs/ft]')
axes[2].set_title('Bias by Formation\n(+ = overestimates DT | − = underestimates DT)')
axes[2].grid(alpha=0.3, axis='x')

fig.suptitle(f'{MODEL_NAME} — Performance by Geological Formation\n'
             f'Bacia Sergipe-Alagoas | LOWO', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'performance_by_formation.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.2 Formations in Poor-Performing Wells

Are the 4 wells with R² < 0.70 in distinct depositional environments?
For each well, we compute the proportion of the analyzed interval within each formation.

In [ ]:
# POOR_WELLS = ['3-BRSA-1069-SES', '3-BRSA-1207-SES',
#              '1-BRSA-930D-SES',  '1-BRSA-1013-SES']

# Proportion of each formation in each well
form_by_well = (df_valid.groupby(['Well_Name', 'Formation']).size().reset_index(name='N')
)
total_by_well = df_valid.groupby('Well_Name')['Formation'].count().reset_index(name='Total')
form_by_well  = form_by_well.merge(total_by_well, on='Well_Name')
form_by_well['Pct'] = (form_by_well['N'] / form_by_well['Total'] * 100).round(1)
form_by_well = form_by_well.merge(df_metr[['Well_Name', 'R2']], on='Well_Name')
form_by_well = form_by_well.sort_values(['R2', 'Pct'], ascending=[True, False])

print('Formations in Poor-Performing Wells (R² < 0.70)')
for well in POOR_WELLS:
    r2  = df_metr.loc[df_metr['Well_Name']==well, 'R2'].values[0]
    sub = form_by_well[form_by_well['Well_Name']==well]
    print(f'\n{well} (R²={r2:.3f}):')
    for _, row in sub.iterrows():
        print(f'  {row["Formation"]:25s}: {row["N"]:5,} samples ({row["Pct"]:5.1f}%)')

print('\nFormations in High-Performing Wells (R² ≥ 0.90)')
good_wells = df_metr[df_metr['R2'] >= 0.90]['Well_Name'].tolist()
for well in good_wells:
    r2  = df_metr.loc[df_metr['Well_Name']==well, 'R2'].values[0]
    sub = form_by_well[form_by_well['Well_Name']==well]
    forms = ', '.join([f"{r['Formation'].replace('FM.','').strip()} ({r['Pct']:.0f}%)"
                       for _, r in sub.iterrows()])
    print(f'  {well} (R²={r2:.3f}): {forms}')


In [ ]:
# Stacked bar: formation composition per well, sorted by R²
wells_sorted = df_metr.sort_values('R2')['Well_Name'].tolist()

pivot_form = (
    form_by_well.pivot_table(
        index='Well_Name', columns='Formation', values='Pct', fill_value=0
    ).reindex(wells_sorted)
)

fig, ax = plt.subplots(figsize=(12, 11))
bottom = np.zeros(len(pivot_form))
for form in pivot_form.columns:
    vals  = pivot_form[form].values
    color = FORMATION_COLORS.get(form, '#BDC3C7')
    ax.barh(pivot_form.index, vals, left=bottom,
            color=color, edgecolor='white', linewidth=0.5,
            label=form.replace('FM. ', ''))
    bottom += vals

# Threshold line
n_poor = len(POOR_WELLS)
ax.axhline(n_poor - 0.5, color='red', linewidth=2.0, linestyle='--')

# R²
for i, well in enumerate(wells_sorted):
    r2        = df_metr.loc[df_metr['Well_Name'] == well, 'R2'].values[0]
    is_poor   = well in POOR_WELLS
    txt_color = 'red' if is_poor else 'black'
    weight    = 'bold' if is_poor else 'normal'
    ax.text(50, i, f'R²={r2:.3f}', va='center', ha='center',
            fontsize=7.5, color=txt_color, fontweight=weight,
            bbox=dict(facecolor='white', alpha=0.55, edgecolor='none', pad=1.5))

ax.set_xlabel('Proportion of analyzed interval (%)', fontsize=11)
ax.set_title(f'{MODEL_NAME} — Formation Composition per Well\n'
             f'Sorted by R² LOWO (worst → best) | red line = threshold R²<0.70',
             fontsize=12)
ax.set_xlim(0, 100)

# Legend
ax.legend(
    title='Formation', fontsize=9, title_fontsize=9,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.06),
    ncol=5,
    framealpha=0.9,
    edgecolor='gray'
)

ax.grid(alpha=0.3, axis='x')
plt.subplots_adjust(bottom=0.10)
plt.savefig(FIGURES_PATH / 'formation_composition_by_well.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.3 Error by Formation — Distribution and Depth

In [ ]:
# Residuals boxplot + MAE by depth per formation
formations = [f for f in df_form_metrics.sort_values('R2')['Formation'].tolist()
              if f != 'out_of_range']

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Boxplot residuals by formation
data_box = [df_valid[df_valid['Formation']==f]['residual'].values for f in formations]
bp = axes[0].boxplot(data_box, vert=False, patch_artist=True,
                     medianprops={'color': 'black', 'linewidth': 2},
                     flierprops={'marker': '.', 'markersize': 2, 'alpha': 0.3})
for patch, form in zip(bp['boxes'], formations):
    patch.set_facecolor(FORMATION_COLORS.get(form, '#BDC3C7'))
    patch.set_alpha(0.85)
axes[0].axvline(0, color='red', linestyle='--', lw=1.2)
axes[0].set_yticks(range(1, len(formations)+1))
axes[0].set_yticklabels([f.replace('FM. ', '') for f in formations], fontsize=9)
axes[0].set_xlabel('Residual (DT_pred − DT_real) [µs/ft]')
axes[0].set_title('Residual Distribution by Formation')
axes[0].set_xlim(-40, 40)
axes[0].grid(alpha=0.3)

# MAE per depth bin by formation
DEPTH_BIN = 500
for form in formations:
    sub = df_valid[df_valid['Formation']==form].copy()
    if len(sub) < 100: continue
    sub['bin'] = (sub['DEPTH'] // DEPTH_BIN * DEPTH_BIN).astype(int)
    trend = sub.groupby('bin')['abs_error'].mean().reset_index()
    if len(trend) >= 2:
        axes[1].plot(trend['abs_error'], trend['bin'],
                     color=FORMATION_COLORS.get(form, '#BDC3C7'),
                     linewidth=2, marker='o', markersize=5,
                     label=form.replace('FM. ', ''))

axes[1].invert_yaxis()
axes[1].set_xlabel('Mean MAE [µs/ft]')
axes[1].set_ylabel('Depth [m]')
axes[1].set_title(f'MAE by Depth ({DEPTH_BIN}m bins) by Formation')
axes[1].legend(fontsize=9, loc='lower right')
axes[1].grid(alpha=0.3)

fig.suptitle(f'{MODEL_NAME} — Error by Formation | Sergipe-Alagoas Basin', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'error_by_formation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Detailed numerical metrics: well × formation × lithology × depth
DEPTH_BIN_SIZE = 200  # meters

df_analysis = df[df['Formation'] != 'out_of_range'].copy()
df_analysis['DEPTH_bin'] = (df_analysis['DEPTH'] // DEPTH_BIN_SIZE * DEPTH_BIN_SIZE).astype(int)

# 1. By well × formation
by_well_form = []
for (well, form), grp in df_analysis.groupby(['Well_Name', 'Formation']):
    if len(grp) < 10:
        continue
    by_well_form.append({
        'Well_Name' : well,
        'Formation' : form,
        'R2_well'   : grp['R2'].iloc[0],
        'N'         : len(grp),
        'MAE'       : grp['abs_error'].mean(),
        'RMSE'      : np.sqrt((grp['residual']**2).mean()),
        'Bias'      : grp['residual'].mean(),
        'P50_err'   : grp['abs_error'].quantile(0.50),
        'P90_err'   : grp['abs_error'].quantile(0.90),
        'DEPTH_min' : grp['DEPTH'].min(),
        'DEPTH_max' : grp['DEPTH'].max(),
        'Is_poor'   : well in POOR_WELLS,
    })

df_wf = pd.DataFrame(by_well_form).sort_values(['R2_well', 'Formation'])
print('By Well × Formation')
print(df_wf.to_string(index=False, float_format='{:.3f}'.format))

# 2. By well × lithology (poor-performing wells only)
print('\n\n By Well × Lithology (poor-performing wells)')
by_well_lith = []
for (well, lith), grp in df_analysis[df_analysis['Well_Name'].isin(POOR_WELLS)]\
                          .groupby(['Well_Name', 'Lithology_norm']):
    if len(grp) < 10:
        continue
    by_well_lith.append({
        'Well_Name' : well,
        'R2_well'   : grp['R2'].iloc[0],
        'Lithology' : lith,
        'N'         : len(grp),
        'MAE'       : grp['abs_error'].mean(),
        'Bias'      : grp['residual'].mean(),
        'P90_err'   : grp['abs_error'].quantile(0.90),
        'DEPTH_min' : grp['DEPTH'].min(),
        'DEPTH_max' : grp['DEPTH'].max(),
    })

df_wl = pd.DataFrame(by_well_lith).sort_values(['R2_well', 'MAE'], ascending=[True, False])
print(df_wl.to_string(index=False, float_format='{:.3f}'.format))

# 3. By well × depth bin (poor-performing wells)
print('\n\n By Well × Depth Bin (poor-performing wells)')
by_well_depth = []
for (well, bin_d), grp in df_analysis[df_analysis['Well_Name'].isin(POOR_WELLS)]\
                           .groupby(['Well_Name', 'DEPTH_bin']):
    if len(grp) < 10:
        continue
    by_well_depth.append({
        'Well_Name'  : well,
        'R2_well'    : grp['R2'].iloc[0],
        'DEPTH_bin'  : bin_d,
        'N'          : len(grp),
        'MAE'        : grp['abs_error'].mean(),
        'Bias'       : grp['residual'].mean(),
        'P90_err'    : grp['abs_error'].quantile(0.90),
        'Formation'  : grp['Formation'].mode()[0],
        'Lith_dom'   : grp['Lithology_norm'].mode()[0],
    })

df_wd = pd.DataFrame(by_well_depth).sort_values(['Well_Name', 'DEPTH_bin'])
print(df_wd.to_string(index=False, float_format='{:.3f}'.format))

# 4. Feature × error correlation by poor-performing well
print('\n\nSpearman Correlation: Features × Absolute Error (poor-performing wells)')
features = ['GR', 'RT90', 'RHOB', 'NPHI', 'DEPTH']
for well in POOR_WELLS:
    sub = df_analysis[df_analysis['Well_Name'] == well]
    r2  = sub['R2'].iloc[0]
    print(f'\n{well} (R²={r2:.3f}):')
    for feat in features:
        if feat not in sub.columns:
            continue
        rho, p = spearmanr(sub[feat], sub['abs_error'])
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
        print(f'  {feat:8s}: rho={rho:+.3f}  p={p:.4f}  {sig}')

### 7.4 MAE Heatmap — Formation × Lithology

Identifies geological contexts with the highest and lowest prediction error.

In [ ]:
# Heatmap: mean MAE by formation × lithology
pivot_mae = (
    df_valid.groupby(['Formation', 'Lithology_norm'])['abs_error']
    .mean().unstack(fill_value=np.nan).round(2)
)

# Filter lithologies with at least 100 samples
lith_counts = df_valid.groupby('Lithology_norm').size()
valid_liths = lith_counts[lith_counts >= 100].index.tolist()
pivot_mae   = pivot_mae[[c for c in pivot_mae.columns if c in valid_liths]]

norm = mcolors.Normalize(vmin=0, vmax=10)
cmap_h = plt.cm.RdYlGn_r

fig, ax = plt.subplots(figsize=(max(8, len(pivot_mae.columns)*1.2), 5))

for i, form in enumerate(pivot_mae.index):
    for j, lith in enumerate(pivot_mae.columns):
        val   = pivot_mae.loc[form, lith]
        color = '#EEEEEE' if pd.isna(val) else cmap_h(norm(val))
        txt   = 'N/A' if pd.isna(val) else f'{val:.1f}'
        rect  = plt.Rectangle((j-0.5, i-0.5), 1, 1,
                              facecolor=color, edgecolor='white', lw=0.8)
        ax.add_patch(rect)
        txt_color = 'white' if (not pd.isna(val) and val > 7) else 'black'
        ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                color=txt_color, fontweight='bold')

ax.set_xlim(-0.5, len(pivot_mae.columns)-0.5)
ax.set_ylim(-0.5, len(pivot_mae.index)-0.5)
ax.set_xticks(range(len(pivot_mae.columns)))
ax.set_xticklabels([l.replace('_', ' ').capitalize() for l in pivot_mae.columns],
                   rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot_mae.index)))
ax.set_yticklabels([f.replace('FM. ', '') for f in pivot_mae.index], fontsize=9)
sm = plt.cm.ScalarMappable(cmap=cmap_h, norm=norm)
plt.colorbar(sm, ax=ax, label='MAE [µs/ft]', shrink=0.8)
ax.set_title(f'{MODEL_NAME} — Mean MAE by Formation × Lithology\n'
             f'(N/A = fewer than 100 samples in the combination)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'heatmap_formation_lithology.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTable: Mean MAE by Formation × Lithology')
print(pivot_mae.to_string())


### 7.5 Bubble Chart: Bias × MAE by Formation × Lithology

Classifies each formation×lithology combination into four quadrants: high/low error × overestimation/underestimation of DT.

In [ ]:
# Bubble chart: mean MAE by formation × lithology
# ── Bubble chart: Bias × MAE by Formation × Lithology ──────────────────────────
bubble_data = (
    df[df['Formation'] != 'out_of_range']
    .groupby(['Formation', 'Lithology_norm'])
    .agg(
        N       = ('abs_error', 'count'),
        MAE     = ('abs_error', 'mean'),
        Bias    = ('residual',  'mean'),
        P90_err = ('abs_error', lambda x: x.quantile(0.90)),
    )
    .reset_index()
    .query('N >= 100')
    .copy()
)

bubble_data['bubble_size'] = np.log10(bubble_data['N']) *120

fig, ax = plt.subplots(figsize=(14, 11))

texts = []
for form in bubble_data['Formation'].unique():
    sub   = bubble_data[bubble_data['Formation'] == form]
    color = FORMATION_COLORS.get(form, '#BDC3C7')
    ax.scatter(sub['Bias'], sub['MAE'],
               s=sub['bubble_size'], color=color, alpha=0.75,
               edgecolors='black', linewidth=0.8, zorder=3,
               label=form.replace('FM. ', ''))
    for _, row in sub.iterrows():
        # Initial offset places text outside the bubble before repulsion
        r_approx = np.sqrt(row['bubble_size'] / np.pi) * 0.012
        t = ax.text(
            row['Bias'] + r_approx,
            row['MAE']  + r_approx,
            row['Lithology_norm'].replace('_', ' '),
            fontsize=7.5, ha='left', va='bottom',
            fontweight='bold', color='black', zorder=5,
            bbox=dict(facecolor='white', alpha=0.6,
                      edgecolor='none', pad=1.0)
        )
        texts.append(t)

adjust_text(
    texts, ax=ax,
    arrowprops=dict(arrowstyle='-', color='gray', lw=0.6, alpha=0.7),
    expand_points=(2.5, 2.5),
    expand_text=(1.4, 1.4),
    force_text=(1.2, 1.2),
    force_points=(1.5, 1.5),
    only_move={'points': 'xy', 'texts': 'xy'},
    avoid_self=True,
)

xlim     = max(abs(bubble_data['Bias'].min()),
               abs(bubble_data['Bias'].max())) * 1.3
ylim_max = bubble_data['MAE'].max() * 1.25
mae_mean = bubble_data['MAE'].mean()

ax.set_xlim(-xlim, xlim)
ax.set_ylim(0, ylim_max)
ax.axvline(0, color='black', lw=1.2, linestyle='--', alpha=0.6)
ax.axhline(mae_mean, color='gray', lw=1.0, linestyle=':', alpha=0.6)
ax.fill_betweenx([mae_mean, ylim_max], -xlim, 0,
                 color='#2980B9', alpha=0.05)
ax.fill_betweenx([mae_mean, ylim_max], 0, xlim,
                 color='#E74C3C', alpha=0.05)

for x, ha, txt, cor, va, y in [
    (-xlim*0.97, 'left',  'High error\nUnderestimates DT\nHigh velocity',   '#2980B9', 'top',    ylim_max*0.98),
    ( xlim*0.97, 'right', 'High error\nOverestimates DT\nLow velocity', '#E74C3C', 'top',    ylim_max*0.98),
    (-xlim*0.97, 'left',  'Low error\nUnderestimates DT\nHigh velocity',   '#2980B9', 'bottom', ylim_max*0.04),
    ( xlim*0.97, 'right', 'Low error\nOverestimates DT\nLow velocity', '#E74C3C', 'bottom', ylim_max*0.04),
]:
    ax.text(x, y, txt, ha=ha, va=va, fontsize=8, color=cor, alpha=0.7)

ax.set_xlabel('Bias [µs/ft]   (+ = overestimates DT  |  − = underestimates DT)',
              fontsize=11, fontweight='bold')
ax.set_ylabel('MAE [µs/ft]', fontsize=11, fontweight='bold')
ax.set_title(f'{MODEL_NAME} — Bias × MAE by Formation × Lithology\n'
             f'Bubble size ∝ log(N samples) | N ≥ 100',
             fontsize=12, fontweight='bold')
ax.grid(alpha=0.25)

form_handles = [
    mpatches.Patch(color=FORMATION_COLORS.get(f, '#BDC3C7'),
                   label=f.replace('FM. ', ''))
    for f in bubble_data['Formation'].unique()
]
size_handles = [
    ax.scatter([], [], s=np.log10(n)*120, color='gray',
               alpha=0.6, edgecolors='black', lw=0.8, label=f'N={n:,}')
    for n in [100, 1_000, 10_000, 100_000]
]
l1 = ax.legend(handles=form_handles, title='Formation', fontsize=9,
               title_fontsize=9, loc='upper center',
               bbox_to_anchor=(0.5, -0.08), ncol=5,
               framealpha=0.9, edgecolor='gray')
ax.add_artist(l1)
ax.legend(handles=size_handles, title='N samples', fontsize=8,
          title_fontsize=8, loc='upper center',
          bbox_to_anchor=(0.5, -0.15), ncol=4,
          framealpha=0.9, edgecolor='gray')

plt.subplots_adjust(bottom=0.22)
plt.savefig(FIGURES_PATH / 'bubble_formation_lithology.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Quadrant classification for analysis
mae_mean = bubble_data['MAE'].mean()

def quadrante(row):
    alto  = row['MAE']  > mae_mean
    super_ = row['Bias'] > 0
    if   alto and     super_: return '❌ High error + Overestimation'
    elif alto and not super_: return '❌ High error + Underestimation'
    elif not alto and super_: return '⚠️ Low error + Overestimation'
    else:                     return '✅ Low error + Underestimation'

bubble_data['Quadrante'] = bubble_data.apply(quadrante, axis=1)

print(f'Mean MAE global: {mae_mean:.3f} µs/ft\n')
for q in ['❌ High error + Overestimation',
           '❌ High error + Underestimation',
           '⚠️ Low error + Overestimation',
           '✅ Low error + Underestimation']:
    sub = bubble_data[bubble_data['Quadrante'] == q]\
          .sort_values('MAE', ascending=False)
    if sub.empty:
        continue
    print(f'{q}')
    print(sub[['Formation','Lithology_norm','N','MAE','Bias','P90_err']]
          .rename(columns={'Lithology_norm':'Lithology',
                           'Formation':'Fm.'})
          .to_string(index=False, float_format='{:.2f}'.format))
    print()

## 8. Hypothesis H5 — Lithology

> **Hypothesis:** Prediction quality varies systematically across lithologies due to
> differences in acoustic impedance, cementation, and mineralogical composition that
> are not fully distinguishable from the input features alone.

> **Method:** Per-lithology metrics (R², MAE, Bias, RMSE); residual boxplots; directional
> bias analysis; scatter N×R² to test confounding by sample count; scatter Depth×MAE
> by lithology to identify depth-dependent lithology effects; numerical cross-tabulation
> by formation × lithology × depth bin.

### 8.1 Lithology Analysis

In [ ]:
# Residuals boxplot + R² bars — all lithologies
lith_order = df_lith.sort_values('R2')['Lithology'].tolist()

# Visual representativity marker in Y-axis labels
# Using simple characters compatible with matplotlib
repr_icon = {
    '✅ High'       : ' [H]',
    '⚠️ Moderate'  : ' [M]',
    '🔶 Low'       : ' [B]',
    '❌ Very Low'  : ' [*]',
}

def label_with_repr(lith):
    row = df_lith[df_lith['Lithology'] == lith]
    if len(row) == 0:
        return lith.replace('_', ' ').capitalize()
    icon = repr_icon.get(row['Representatividade'].values[0], '')
    return lith.replace('_', ' ').capitalize() + icon

df_box    = df[df['Lithology_norm'].isin(lith_order)].copy()
r2_global = r2_score(df['DT_real'], df['DT_pred'])

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(lith_order) * 0.65)))

# Boxplot residuals by lithology
data_box = [df_box[df_box['Lithology_norm'] == l]['residual'].values
            for l in lith_order]
bp = axes[0].boxplot(data_box, vert=False, patch_artist=True,
                     medianprops={'color': 'black', 'linewidth': 2},
                     flierprops={'marker': '.', 'markersize': 2, 'alpha': 0.3})
for patch, lith in zip(bp['boxes'], lith_order):
    patch.set_facecolor(lith_color(lith))
    patch.set_alpha(0.85)

axes[0].axvline(0, color='red', linestyle='--', linewidth=1.2, label='Residual = 0')
axes[0].set_yticks(range(1, len(lith_order) + 1))
axes[0].set_yticklabels([label_with_repr(l) for l in lith_order], fontsize=9)
axes[0].set_xlabel('Residual (DT_pred − Measured DT) [µs/ft]')
axes[0].set_title('Residual Distribution by Lithology')
axes[0].legend(fontsize=9)
axes[0].set_xlim(-60, 60)

# Bars: R² by lithology
r2_vals    = [df_lith.loc[df_lith['Lithology'] == l, 'R2'].values[0] for l in lith_order]
bar_colors = [lith_color(l) for l in lith_order]

bars = axes[1].barh([label_with_repr(l) for l in lith_order],
                    r2_vals, color=bar_colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, r2_vals):
    axes[1].text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                 f'{val:.3f}', va='center', fontsize=9)

axes[1].axvline(r2_global, color='black', linestyle='--', linewidth=1.2,
                label=f'R² global = {r2_global:.3f}')
axes[1].set_xlabel('R²')
axes[1].set_title('R² by Lithology\n([H]=High | [M]=Moderate | [B]=Low | [*]=Very Low representativity)')
axes[1].set_xlim(0, 1.15)
axes[1].legend(fontsize=9)

plt.suptitle(f'{MODEL_NAME} — Lithology Analysis | Sergipe-Alagoas Basin',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'lithology_residuals_r2.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Global R² (all wells): {r2_global:.4f}')

In [ ]:
# Directional bias by lithology
print('Bias by Lithology (DT_pred − DT_real)\n')
print('Positive = model OVERESTIMATES DT (velocity underestimated)')
print('Negative = model UNDERESTIMATES DT (velocity overestimated)')
print()
for _, row in df_lith.sort_values('Bias').iterrows():
    tag  = '↑ overestimates' if row['Bias'] > 0.5 else ('↓ underestimates' if row['Bias'] < -0.5 else '≈ neutral')
    repr_tag = row['Representatividade']
    print(f"{row['Lithology']:25s} | Bias = {row['Bias']:+.2f} µs/ft | {tag:15s} | N = {row['N_samples']:>6,} | {repr_tag}")

In [ ]:
# Scatter: N samples × R² (visualize confounding)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors_lith = [lith_color(l) for l in df_lith['Lithology']]

# R² vs N samples
axes[0].scatter(df_lith['N_samples'], df_lith['R2'],
                c=colors_lith, s=80, edgecolors='black', linewidth=0.5, zorder=3)
for _, row in df_lith.iterrows():
    axes[0].annotate(row['Lithology'].replace('_', '\n'),
                     (row['N_samples'], row['R2']),
                     fontsize=6.5, ha='left', va='bottom',
                     xytext=(4, 2), textcoords='offset points')
axes[0].set_xscale('log')
axes[0].set_xlabel('N samples (log scale)', fontsize=11)
axes[0].set_ylabel('R²', fontsize=11)
axes[0].set_title('R² vs N Samples by Lithology\n(log scale — representativity confounding)')
axes[0].axhline(r2_global, color='black', linestyle='--', linewidth=1,
                label=f'R² global = {r2_global:.3f}')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# R² vs N wells
axes[1].scatter(df_lith['N_wells'], df_lith['R2'],
                c=colors_lith, s=80, edgecolors='black', linewidth=0.5, zorder=3)
for _, row in df_lith.iterrows():
    axes[1].annotate(row['Lithology'].replace('_', '\n'),
                     (row['N_wells'], row['R2']),
                     fontsize=6.5, ha='left', va='bottom',
                     xytext=(4, 2), textcoords='offset points')
axes[1].set_xlabel('N distinct wells', fontsize=11)
axes[1].set_ylabel('R²', fontsize=11)
axes[1].set_title('R² vs N Wells by Lithology\n(LOWO generalization)')
axes[1].axhline(r2_global, color='black', linestyle='--', linewidth=1)
axes[1].grid(alpha=0.3)

plt.suptitle(f'{MODEL_NAME} — R² by Lithology vs Training Representativity', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'lithology_r2_vs_representativity.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.2 Depth × MAE by Lithology

Intersection between lithology, depth, and error — identifies whether certain lithologies are harder to predict in shallow or deep zones.

In [ ]:
# Scatter: Depth vs MAE by lithology — points colored by formation
df_scatter = df[df['Lithology_norm'].isin(valid_liths_main)].copy()

n_liths = len(valid_liths_main)
n_cols  = 4
n_rows  = (n_liths + n_cols - 1) // n_cols

# Formation legend (global, outside subplots)
form_handles = [
    mpatches.Patch(color=FORMATION_COLORS[f], label=f.replace('FM. ', ''))
    for f in FORMATION_COLORS if f != 'out_of_range'
]

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 7 * n_rows))
axes_flat = axes.flatten()

for i, (ax, lith) in enumerate(zip(axes_flat, valid_liths_main)):
    sub = df_scatter[df_scatter['Lithology_norm'] == lith].copy()

    # Stratified subsample by formation to maintain representativity
    if len(sub) > 5000:
        sub = (sub.groupby('Formation', group_keys=False)
                  .apply(lambda x: x.sample(
                      min(len(x), max(1, int(5000 * len(x) / len(sub)))),
                      random_state=42))
               )

    # Scatter colored by formation
    for form, grp in sub.groupby('Formation'):
        color = FORMATION_COLORS.get(form, '#BDC3C7')
        ax.scatter(grp['abs_error'], grp['DEPTH'],
                   color=color, alpha=0.3, s=6, zorder=2)

    # Overall mean trend (all points, regardless of formation)
    sub['DEPTH_bin'] = (sub['DEPTH'] // DEPTH_BIN_SIZE * DEPTH_BIN_SIZE).astype(int)
    trend = (sub.groupby('DEPTH_bin')
               .agg(MAE_mean=('abs_error', 'mean'), N=('abs_error', 'count'))
               .reset_index()
               .query('N >= 10'))
    if len(trend) >= 2:
        ax.plot(trend['MAE_mean'], trend['DEPTH_bin'],
                color='black', linewidth=2, linestyle='--', zorder=5,
                label='Mean trend')

    ax.invert_yaxis()
    ax.set_xlim(0, 30)
    n_total = df_scatter[df_scatter['Lithology_norm'] == lith].shape[0]
    ax.set_title(f'{lith.replace("_", " ").capitalize()}\n(N={n_total:,})', fontsize=10)
    ax.set_xlabel('MAE [µs/ft]', fontsize=9)
    ax.grid(alpha=0.3)
    if i % n_cols == 0:
        ax.set_ylabel('Depth [m]', fontsize=10)

# Hide empty subplots
for ax in axes_flat[n_liths:]:
    ax.set_visible(False)

# Global formation legend
fig.legend(handles=form_handles, loc='upper right',
           bbox_to_anchor=(0.98, 0.98), fontsize=9,
           title='Formation', title_fontsize=9,
           framealpha=0.9, ncol=1)

plt.suptitle(f'{MODEL_NAME} — Absolute Error vs Depth by Lithology\n'
             f'Points colored by Formation | dashed line = mean trend',
             fontsize=13, x=0.45)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'scatter_error_depth_lithology_formation_30.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Numerical metrics: formation × lithology × depth bin
DEPTH_BIN_SIZE = 500

df_analysis = df[df['Formation'] != 'out_of_range'].copy()
df_analysis['DEPTH_bin'] = (df_analysis['DEPTH'] // DEPTH_BIN_SIZE * DEPTH_BIN_SIZE).astype(int)

results = []
for (form, lith, bin_d), grp in df_analysis.groupby(
        ['Formation', 'Lithology_norm', 'DEPTH_bin']):
    if len(grp) < 30:
        continue
    results.append({
        'Formation' : form,
        'Lithology' : lith,
        'DEPTH_bin' : bin_d,
        'N'         : len(grp),
        'N_wells'   : grp['Well_Name'].nunique(),
        'MAE'       : grp['abs_error'].mean(),
        'Bias'      : grp['residual'].mean(),
        'P50_err'   : grp['abs_error'].quantile(0.50),
        'P90_err'   : grp['abs_error'].quantile(0.90),
        'DT_mean'   : grp['DT_real'].mean(),
        'DT_std'    : grp['DT_real'].std(),
    })

df_flb = pd.DataFrame(results).sort_values(
    ['Formation', 'Lithology', 'DEPTH_bin']
)

# Print by formation
for form, grp_f in df_flb.groupby('Formation'):
    print(f'\n{"="*75}')
    print(f'FORMATION: {form}')
    print(f'{"="*75}')
    print(grp_f.drop(columns='Formation').to_string(
        index=False, float_format='{:.2f}'.format
    ))

# Summary: mean MAE by formation × lithology (collapsing depth)
print('\n\nSUMMARY: Mean MAE by Formation × Lithology')
summary = (
    df_analysis.groupby(['Formation', 'Lithology_norm'])
    .agg(
        N         =('abs_error', 'count'),
        N_wells   =('Well_Name', 'nunique'),
        MAE       =('abs_error', 'mean'),
        Bias      =('residual',  'mean'),
        P90_err   =('abs_error', lambda x: x.quantile(0.90)),
        DT_mean   =('DT_real',   'mean'),
        DT_std    =('DT_real',   'std'),
    )
    .reset_index()
    .query('N >= 30')
    .sort_values(['Formation', 'MAE'], ascending=[True, False])
)
print(summary.to_string(index=False, float_format='{:.3f}'.format))

## 9. Hypothesis H6 — Layer Thickness

> **Hypothesis:** Thinner lithological layers increase local DT variability within short
> depth intervals, making prediction harder. A Spearman correlation between layer thickness
> and absolute error tests whether this effect has quantitative support in the data.

> **Method:** Bin samples by layer thickness (<5m, 5–15m, 15–30m, 30–60m, 60–100m, >100m);
> compute MAE and Bias per bin; analyze by lithology and poor-performing wells;
> compute per-well Spearman correlation between layer thickness and absolute error.

In [ ]:
# Layer thickness × prediction error
# Thickness available in Lithology_Segment_Thickness (merged from df_main)
# and in df_segments via Notebook 03

# Verify column availability
print('Columns with thickness:')
thick_cols = [c for c in df.columns if 'thick' in c.lower() or 'segment' in c.lower()]
print(thick_cols)
print(f'\nSamples with thickness available: {df["Lithology_Segment_Thickness"].notna().sum():,}')
print(f'Samples without thickness: {df["Lithology_Segment_Thickness"].isna().sum():,}')
print()

df_thick = df[df['Lithology_Segment_Thickness'].notna()].copy()

# 1. Global correlation: thickness × error
rho_global, p_global = spearmanr(
    df_thick['Lithology_Segment_Thickness'],
    df_thick['abs_error']
)
print(f'Global Spearman Correlation: Thickness × Absolute Error')
print(f'ρ = {rho_global:+.4f}  (p = {p_global:.6f})')
print()

# 2. Thickness ranges — error metrics
bins_thick   = [0, 5, 15, 30, 60, 100, np.inf]
labels_thick = ['<5m', '5–15m', '15–30m', '30–60m', '60–100m', '>100m']

df_thick['Thick_bin'] = pd.cut(
    df_thick['Lithology_Segment_Thickness'],
    bins=bins_thick, labels=labels_thick
)

thick_stats = (
    df_thick.groupby('Thick_bin', observed=True)
    .agg(
        N           =('abs_error', 'count'),
        N_segments  =('Lithology_Segment_ID', 'nunique'),
        MAE         =('abs_error', 'mean'),
        Bias        =('residual',  'mean'),
        P50_err     =('abs_error', lambda x: x.quantile(0.50)),
        P90_err     =('abs_error', lambda x: x.quantile(0.90)),
        DT_std      =('DT_real',   'std'),
        Thick_mean  =('Lithology_Segment_Thickness', 'mean'),
    )
    .reset_index()
)

print('Metrics by Thickness Range')
print(thick_stats.to_string(index=False, float_format='{:.3f}'.format))

# 3. By lithology × thickness range
print('\n\nMAE by Lithology × Thickness Range')
lith_thick = (
    df_thick.groupby(['Lithology_norm', 'Thick_bin'], observed=True)
    .agg(
        N       =('abs_error', 'count'),
        MAE     =('abs_error', 'mean'),
        Bias    =('residual',  'mean'),
        P90_err =('abs_error', lambda x: x.quantile(0.90)),
    )
    .reset_index()
    .query('N >= 50')
    .sort_values(['Lithology_norm', 'Thick_bin'])
)
print(lith_thick.to_string(index=False, float_format='{:.3f}'.format))

# 4. By poor-performing well × thickness range
print('\n\nMAE by Poor-Performing Well × Thickness Range')
poor_thick = (
    df_thick[df_thick['Well_Name'].isin(POOR_WELLS)]
    .groupby(['Well_Name', 'Thick_bin'], observed=True)
    .agg(
        N       =('abs_error', 'count'),
        MAE     =('abs_error', 'mean'),
        Bias    =('residual',  'mean'),
        P90_err =('abs_error', lambda x: x.quantile(0.90)),
    )
    .reset_index()
    .query('N >= 10')
    .sort_values(['Well_Name', 'Thick_bin'])
)
print(poor_thick.to_string(index=False, float_format='{:.3f}'.format))

# 5. Correlation by poor-performing well
print('\n\n Spearman Correlation: Thickness × Error by Poor-Performing Well')
for well in POOR_WELLS:
    sub = df_thick[df_thick['Well_Name'] == well]
    if len(sub) < 30:
        continue
    rho, p = spearmanr(sub['Lithology_Segment_Thickness'], sub['abs_error'])
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    r2  = sub['R2'].iloc[0]
    print(f'  {well} (R²={r2:.3f}): ρ={rho:+.4f}  p={p:.4f}  {sig}  N={len(sub):,}')

# 6. Comparison: poor-performing vs good wells
print('\n\n Mean layer thickness: poor-performing vs good')
df_thick['Group'] = df_thick['Well_Name'].apply(
    lambda w: 'Poor-performing' if w in POOR_WELLS else 'Good (R²≥0.75)'
    if df_metr.loc[df_metr['Well_Name']==w, 'R2'].values[0] >= 0.75
    else 'Moderate'
)
print(df_thick.groupby('Group')['Lithology_Segment_Thickness']
      .agg(['mean', 'median', 'std', 'count'])
      .round(2).to_string())

## 10. Hypothesis H7 — Well Directionality

> **Hypothesis:** Deviated wells (inclination > 5°) show lower prediction performance
> because the logged depth does not correspond to true vertical depth, introducing
> systematic offset in the depth-dependent features.

> **Method:** Classify wells by directionality (vertical ≤5° vs deviated); compute
> mean R² per group; Mann-Whitney U test for statistical significance; geographic
> performance map; well trajectory chart referenced by depth.

In [ ]:
# Detailed per-well table
well_detail = (
    df.groupby('Well_Name')
    .agg(DEPTH_mean    =('DEPTH', 'mean'),
         DEPTH_min     =('DEPTH', 'min'),
         DEPTH_max     =('DEPTH', 'max'),
         N_samples     =('DT_real', 'count'),
         Lith_dominant =('Lithology_norm',
                          lambda x: x.value_counts().index[0] if len(x.dropna()) > 0 else 'N/A'),
         N_liths       =('Lithology_norm', 'nunique'),
         MAE_well      =('abs_error', 'mean'),
         Bias_well     =('residual',  'mean'))
    .reset_index()
    .merge(df_metr[['Well_Name', 'R2', 'RMSE']], on='Well_Name')
    .merge(df_geographic[['Well_Name', 'Max_INCL', 'Desl_H', 'Dir_Category']],
           on='Well_Name', how='left')
    .sort_values('R2')
)

THRESHOLD_DETAIL = 0.70
piores = well_detail[well_detail['R2'] < THRESHOLD_DETAIL]

cols_show = ['Well_Name', 'R2', 'RMSE', 'MAE_well', 'Bias_well',
             'DEPTH_mean', 'DEPTH_max', 'Lith_dominant', 'N_liths', 'N_samples',
             'Max_INCL', 'Desl_H', 'Dir_Category']

print(f'Wells with R² < {THRESHOLD_DETAIL} ({len(piores)} wells)')
if piores.empty:
    print('No wells below the threshold.')
else:
    print(piores[cols_show].to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# Analysis: R² vs DT variability per well
dt_std = df.groupby('Well_Name')['DT_real'].std().reset_index()
dt_std.columns = ['Well_Name', 'DT_std']

well_detail_ext = well_detail.merge(dt_std, on='Well_Name')

# Poor-performing wells by dual criterion
THRESHOLD_R2   = 0.70
THRESHOLD_RMSE = 6.0

worst_double = well_detail_ext[
    (well_detail_ext['R2'] < THRESHOLD_R2) |
    (well_detail_ext['RMSE'] > THRESHOLD_RMSE)
]

print(f'Poor-performing wells — dual criterion (R²<{THRESHOLD_R2} OR RMSE>{THRESHOLD_RMSE})')
cols = ['Well_Name', 'R2', 'RMSE', 'DT_std', 'Bias_well', 'DEPTH_max', 'Dir_Category']
print(worst_double[cols].to_string(index=False, float_format='{:.3f}'.format))

# Correlação R² × std do DT
rho_std, p_std = spearmanr(well_detail_ext['DT_std'], well_detail_ext['R2'])
print(f'\nSpearman: R² × std(DT) = {rho_std:+.3f} (p={p_std:.4f})')
print('→ Confirms that wells with higher DT variance naturally achieve higher R².')

In [ ]:
#R² vs max depth and N samples — colored by directionality
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Colors and markers by directional category
dir_style = {
    'Vertical (≤5°)'      : {'color': '#2ECC71', 'marker': 'o', 'label': 'Vertical (≤5°)'},
    'Slight deviation (5–20°)' : {'color': '#F39C12', 'marker': 's', 'label': 'Slight deviation (5–20°)'},
    'Directional (>20°)'       : {'color': '#E74C3C', 'marker': '^', 'label': 'Directional (>20°)'},
}

well_plot = well_detail.copy()
well_plot['Dir_Category'] = well_plot['Dir_Category'].fillna('Vertical (≤5°)')
well_plot['Max_INCL']     = well_plot['Max_INCL'].fillna(0.0)

mean_r2   = well_plot['R2'].mean()
mean_line = plt.Line2D([0], [0], color='black', linestyle='--',
                       label=f'R² médio = {mean_r2:.3f}')

for cat, style in dir_style.items():
    sub = well_plot[well_plot['Dir_Category'] == cat]
    if sub.empty:
        continue
    kw = dict(s=70, alpha=0.85, edgecolors='white', linewidth=0.5,
              color=style['color'], marker=style['marker'],
              label=f'{style["label"]} (n={len(sub)})')

    axes[0].scatter(sub['DEPTH_max'], sub['R2'], **kw)
    axes[1].scatter(sub['N_samples'], sub['R2'], **kw)

    for _, row in sub[sub['R2'] < 0.65].iterrows():
        axes[0].annotate(row['Well_Name'].replace('-SES', ''),
                         xy=(row['DEPTH_max'], row['R2']),
                         xytext=(5, 5), textcoords='offset points',
                         fontsize=7, color=style['color'])
        axes[1].annotate(row['Well_Name'].replace('-SES', ''),
                         xy=(row['N_samples'], row['R2']),
                         xytext=(5, 5), textcoords='offset points',
                         fontsize=7, color=style['color'])

for ax in axes:
    ax.axhline(mean_r2, color='black', linestyle='--', linewidth=1)
    ax.grid(alpha=0.3)
    ax.set_ylabel('R² LOWO individual', fontsize=11)

axes[0].set_xlabel('Maximum depth [m]', fontsize=11)
axes[1].set_xlabel('Number of test samples', fontsize=11)

rho_depth, p_depth = spearmanr(well_plot['DEPTH_max'], well_plot['R2'])
rho_n,     p_n     = spearmanr(well_plot['N_samples'],  well_plot['R2'])

axes[0].set_title(f'R² vs Maximum Depth\nSpearman ρ = {rho_depth:+.3f} (p = {p_depth:.3f})')
axes[1].set_title(f'R² vs N Samples\nSpearman ρ = {rho_n:+.3f} (p = {p_n:.3f})')

# Unified legend on right panel
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles + [mean_line], labels + [f'R² médio = {mean_r2:.3f}'],
               fontsize=8, loc='lower left')
axes[1].legend(fontsize=8, loc='lower left')

plt.suptitle(f'{MODEL_NAME} — Factors Associated with Per-Well Performance\n'
             f'(colored by directional category)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'r2_vs_depth_samples.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistical check: mean R² by directional category
print('\nMean R² by directional category')
print(well_plot.groupby('Dir_Category')[['R2', 'Max_INCL']]
      .agg({'R2': ['mean', 'std', 'count'], 'Max_INCL': 'mean'})
      .round(3).to_string())

In [ ]:
# Mann-Whitney U test — Directional vs Vertical wells
df_h7 = df_metr.merge(
    df_dir_summary[['Well_Name', 'Dir_Category']],
    on='Well_Name', how='left'
)

# Wells without directional data → assume vertical
df_h7['Dir_Category'] = df_h7['Dir_Category'].fillna('Vertical (≤5°)')

# Two groups: directional (>20°) vs vertical (≤5°)
r2_vertical    = df_h7[df_h7['Dir_Category'] == 'Vertical (≤5°)']['R2'].values
r2_directional = df_h7[df_h7['Dir_Category'] == 'Directional (>20°)']['R2'].values

stat, p = mannwhitneyu(r2_vertical, r2_directional, alternative='two-sided')

print('H7: Mann-Whitney U Test — Directionality vs R²\n')
print(f'Vertical    (≤5°)  : n={len(r2_vertical)}, mean R²={r2_vertical.mean():.4f}, median={np.median(r2_vertical):.4f}')
print(f'Directional (>20°) : n={len(r2_directional)}, mean R²={r2_directional.mean():.4f}, median={np.median(r2_directional):.4f}')
print(f'ΔR² (Vertical − Directional) = {r2_vertical.mean() - r2_directional.mean():+.4f}')
print(f'Mann-Whitney U = {stat:.1f}, p = {p:.4f}')
if p < 0.05:
    print('→ Statistically significant difference (p < 0.05) — H7 SUPPORTED')
else:
    print('→ Non-significant difference (p ≥ 0.05) — H7 REJECTED')

In [ ]:
# Chart: Well trajectories referenced by depth
df_survey = pd.read_csv('../data/processed/directional_wells.csv')

DEVIATED_THRESHOLD = 10.0
deviated_max_incl = (df_survey.groupby('Well_Name')['INCL'].max()
                     .reset_index().rename(columns={'INCL': 'max_incl'}))
truly_deviated = deviated_max_incl[
    deviated_max_incl['max_incl'] > DEVIATED_THRESHOLD
]['Well_Name'].tolist()

# Available depth range per well
well_depth = (
    df.groupby('Well_Name')['DEPTH']
    .agg(DEPTH_min='min', DEPTH_max='max')
    .reset_index()
    .merge(df_metr[['Well_Name', 'R2']], on='Well_Name')
    .sort_values('R2')
    .reset_index(drop=True)
)
n_wells     = len(well_depth)
X_SPACING   = 1.0
x_positions = {row['Well_Name']: i * X_SPACING
               for i, (_, row) in enumerate(well_depth.iterrows())}

# Horizontal scale factor
max_horiz_all = 0.0
for well in truly_deviated:
    sub   = df_survey[df_survey['Well_Name'] == well]
    horiz = np.sqrt(sub['NS']**2 + sub['EW']**2).max()
    if horiz > max_horiz_all:
        max_horiz_all = horiz
SCALE = 0.38 / max_horiz_all

# Colors
POOR_COLOR = '#E74C3C'
GOOD_COLOR = "#22A311"
TEXT_COLOR = '#404040'

def well_color(r2):
    if r2 < 0.70: return POOR_COLOR
    return GOOD_COLOR

# Figure
fig, ax = plt.subplots(figsize=(22, 11))

for _, row in well_depth.iterrows():
    well  = row['Well_Name']
    r2    = row['R2']
    d_min = row['DEPTH_min']
    d_max = row['DEPTH_max']
    x0    = x_positions[well]
    color = well_color(r2)

    if well in truly_deviated and well in df_survey['Well_Name'].values:
        sub = (df_survey[df_survey['Well_Name'] == well]
               .sort_values('MD').copy())
        sub = sub[(sub['MD'] >= d_min - 50) & (sub['MD'] <= d_max + 50)]
        if sub.empty:
            sub = df_survey[df_survey['Well_Name'] == well].sort_values('MD')

        horiz  = np.sqrt(sub['NS']**2 + sub['EW']**2).values
        tvd    = sub['TVD'].values
        sinal  = 1 if sub['EW'].iloc[-1] >= 0 else -1
        x_traj = x0 + sinal * horiz * SCALE
        y_traj = tvd

        ax.plot(x_traj, y_traj, color=color, lw=2.5,
                solid_capstyle='round', zorder=3)
        ax.scatter(x_traj[0],  y_traj[0],  color=color, s=35, zorder=4)
        ax.scatter(x_traj[-1], y_traj[-1], color=color, s=35,
                   marker='v', zorder=4)

        ax.text(x_traj[0], y_traj[0] - 55,
                f'R²={r2:.3f}',
                ha='center', va='bottom',
                fontsize=12, color=color, fontweight='bold')
        ax.text(x_traj[-1], y_traj[-1] + 60,
                well.replace('-SES', ''),
                ha='center', va='top',
                fontsize=12, color=TEXT_COLOR, rotation=90)

    else:
        ax.plot([x0, x0], [d_min, d_max],
                color=color, lw=2.8,
                solid_capstyle='round', zorder=3)
        ax.scatter(x0, d_min, color=color, s=35, zorder=4)
        ax.scatter(x0, d_max, color=color, s=35, zorder=4)

        ax.text(x0, d_min - 55,
                f'R²={r2:.3f}',
                ha='center', va='bottom',
                fontsize=12, color=color, fontweight='bold')
        ax.text(x0, d_max + 60,
                well.replace('-SES', ''),
                ha='center', va='top',
                fontsize=12, color=TEXT_COLOR, rotation=90)

# Reference lines
DEPTH_TICKS = list(range(2000, 6501, 500))
for depth_ref in DEPTH_TICKS:
    ax.axhline(depth_ref, color='gray', lw=1.0, linestyle=':', alpha=0.8, zorder=1)

ax.invert_yaxis()
ax.set_xlim(-1.2, n_wells + 0.5)
y_max = well_depth['DEPTH_max'].max()
y_min = well_depth['DEPTH_min'].min()
ax.set_ylim(y_max + 700, y_min - 700)
ax.set_xticks([])
ax.set_ylabel('Depth [m]', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.set_yticks(DEPTH_TICKS)
ax.set_yticklabels([f'{d:,}m' for d in DEPTH_TICKS],
                    fontsize=9, color='gray')
ax.tick_params(axis='y', length=0)
ax.grid(True)

legend_elements = [
    plt.Line2D([0], [0], color=POOR_COLOR, lw=2.5,
               label='R² < 0.70 (poor-performing)'),
    plt.Line2D([0], [0], color=GOOD_COLOR, lw=2.5,
               label='R² ≥ 0.75 (good)'),
]
ax.legend(handles=legend_elements, loc='lower right',
          fontsize=11, framealpha=0.9)

ax.set_title(
    f'{MODEL_NAME} — Well Trajectory by Depth\n'
    f'Sorted by R² (left = worst | right = best) | '
    f'Directional wells: trajectory normalized by actual horizontal displacement',
    fontsize=13, fontweight='bold'
)

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'trajectory_wells_depth.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 11. Hypothesis H8 — Petrophysical Regime

> **Hypothesis:** Distinct petrophysical regimes within the same lithological class
> (e.g., Coqueiro Seco sandstone vs other sandstones; Cotinguiba shale vs Calumbi shale)
> cause systematic prediction bias because the model cannot distinguish diagenetically
> or mineralogically differentiated rocks using GR, RT90, RHOB, and NPHI alone.

> **Method:** Compare RHOB, NPHI, DT, GR, and RT90 distributions between formation
> subgroups within the same lithology; visualize separation via histograms and NPHI×DT
> crossplots; quantify percentage differences in medians.

In [ ]:
# Petrophysical comparison: Coqueiro Seco sandstone vs other sandstones
df_h8 = df_main.copy()

# Filter sandstone only
df_ar = df_h8[df_h8['Lithology'] == 'sandstone'].copy()

# Create group: Coqueiro Seco vs others
df_ar['Group'] = df_ar['Formation'].apply(
    lambda x: 'Coqueiro Seco' if x == 'FM. COQUEIRO SECO' else 'Other formations'
)

# Descriptive statistics
cols = ['RHOB', 'NPHI', 'DT', 'GR', 'RT90']
stats = df_ar.groupby('Group')[cols].agg([
    'count', 'mean', 'median', 'std',
    lambda x: x.quantile(0.05),
    lambda x: x.quantile(0.95)
]).round(3)
stats.columns = ['_'.join(col).strip() for col in stats.columns]

print('Petrophysical Comparison: Coqueiro Seco Sandstone vs Others\n')
for col in cols:
    print(f'\n--- {col} ---')
    for group in ['Coqueiro Seco', 'Other formations']:
        sub = df_ar[df_ar['Group'] == group][col].dropna()
        print(f'{group:20s}  N={len(sub):5d}  '
              f'mean={sub.mean():.2f}  '
              f'median={sub.median():.2f}  '
              f'std={sub.std():.2f}  '
              f'P5={sub.quantile(0.05):.2f}  '
              f'P95={sub.quantile(0.95):.2f}')

# Percentage difference in medians
print('\n\nPercentage Difference (Coqueiro Seco vs Others)')
for col in cols:
    med_cs  = df_ar[df_ar['Group']=='Coqueiro Seco'][col].median()
    med_dem = df_ar[df_ar['Group']=='Other formations'][col].median()
    diff_pct = (med_cs - med_dem) / med_dem * 100
    print(f'{col:8s}  Coqueiro Seco={med_cs:.2f}  '
          f'Others={med_dem:.2f}  '
          f'Δ={diff_pct:+.1f}%')

In [ ]:
# Comparison: Coqueiro Seco Sandstone vs Riachuelo Sandstone
df_ar = df_h8[df_h8['Lithology'] == 'sandstone'].copy()
df_ar = df_ar[df_ar['Formation'].isin(['FM. COQUEIRO SECO', 'FM. RIACHUELO'])]
df_ar['Group'] = df_ar['Formation'].str.replace('FM. ', '')

cols = ['RHOB', 'NPHI', 'DT', 'GR', 'RT90']
print('Sandstone: Coqueiro Seco vs Riachuelo\n')
for col in cols:
    for group in ['COQUEIRO SECO', 'RIACHUELO']:
        sub = df_ar[df_ar['Group'] == group][col].dropna()
        print(f'{group:15s}  N={len(sub):5d}  median={sub.median():.2f}  std={sub.std():.2f}')
    cs  = df_ar[df_ar['Group']=='COQUEIRO SECO'][col].median()
    ri  = df_ar[df_ar['Group']=='RIACHUELO'][col].median()
    print(f'  Δ = {(cs-ri)/ri*100:+.1f}%\n')

In [ ]:
# Comparison: Cotinguiba Shale vs Calumbi Shale
df_fo = df_h8[df_h8['Lithology'] == 'shale'].copy()
df_fo = df_fo[df_fo['Formation'].isin(['FM. COTINGUIBA', 'FM. CALUMBI'])]
df_fo['Group'] = df_fo['Formation'].str.replace('FM. ', '')

cols = ['RHOB', 'NPHI', 'DT', 'GR', 'RT90']
print('Shale: Cotinguiba vs Calumbi\n')
for col in cols:
    for group in ['COTINGUIBA', 'CALUMBI']:
        sub = df_fo[df_fo['Group'] == group][col].dropna()
        print(f'{group:12s}  N={len(sub):6d}  median={sub.median():.2f}  std={sub.std():.2f}')
    cot = df_fo[df_fo['Group']=='COTINGUIBA'][col].median()
    cal = df_fo[df_fo['Group']=='CALUMBI'][col].median()
    print(f'  Δ = {(cot-cal)/cal*100:+.1f}%\n')

In [ ]:
# Figure H8c: Petrophysical comparison Coqueiro Seco vs Riachuelo
df_ar = df_h8[df_h8['Lithology'] == 'sandstone'].copy()
df_ar = df_ar[df_ar['Formation'].isin(['FM. COQUEIRO SECO', 'FM. RIACHUELO'])]
df_ar['Group'] = df_ar['Formation'].str.replace('FM. ', '')

colors = {'COQUEIRO SECO': '#d73027', 'RIACHUELO': '#4575b4'}
cols = ['NPHI', 'RHOB', 'DT', 'GR']
labels = {'NPHI': 'NPHI [pu]', 'RHOB': 'RHOB [g/cm³]',
          'DT': 'DT [µs/ft]', 'GR': 'GR [API]'}

fig, axes = plt.subplots(1, 5, figsize=(20, 6))
fig.suptitle('Petrophysical Comparison: Coqueiro Seco vs Riachuelo Sandstone\n(H8c: Diagenetic Mineralogical Control)',
             fontsize=13, fontweight='bold')

# Panels 1–4: distributions by attribute
for i, col in enumerate(cols):
    ax = axes[i]
    for group, color in colors.items():
        sub = df_ar[df_ar['Group'] == group][col].dropna()
        ax.hist(sub, bins=40, alpha=0.6, color=color,
                label=f'{group}\n(N={len(sub):,}, med={sub.median():.1f})',
                density=True, edgecolor='white', linewidth=0.3)
        ax.axvline(sub.median(), color=color, linewidth=2, linestyle='--')
    ax.set_xlabel(labels[col], fontsize=11)
    ax.set_ylabel('Density' if i == 0 else '', fontsize=11)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)

    # Delta annotation
    med_cs = df_ar[df_ar['Group']=='COQUEIRO SECO'][col].median()
    med_ri = df_ar[df_ar['Group']=='RIACHUELO'][col].median()
    delta = (med_cs - med_ri) / med_ri * 100
    ax.text(0.97, 0.97, f'Δ = {delta:+.1f}%',
            transform=ax.transAxes, fontsize=9, ha='right', va='top',
            color='#d73027' if abs(delta) > 10 else '#555555',
            fontweight='bold' if abs(delta) > 10 else 'normal',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Panel 5: Crossplot NPHI × DT
ax = axes[4]
for group, color in colors.items():
    sub = df_ar[df_ar['Group'] == group]
    ax.scatter(sub['NPHI'], sub['DT'], c=color, s=8, alpha=0.4,
               label=group, edgecolors='none')
    # Medians
    ax.scatter(sub['NPHI'].median(), sub['DT'].median(),
               c=color, s=120, marker='D', edgecolors='black',
               linewidths=1, zorder=5)

ax.set_xlabel('NPHI [pu]', fontsize=11)
ax.set_ylabel('DT [µs/ft]', fontsize=11)
ax.set_title('NPHI × DT\nCrossplot', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.text(0.05, 0.95,
        'Diamonds = medians',
        transform=ax.transAxes, fontsize=8, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'coqueiro_vs_riachuelo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Petrophysical comparison Coqueiro Seco vs Riachuelo (2×3 layout)
df_ar = df_h8[df_h8['Lithology'] == 'sandstone'].copy()
df_ar = df_ar[df_ar['Formation'].isin(['FM. COQUEIRO SECO', 'FM. RIACHUELO'])]
df_ar['Group'] = df_ar['Formation'].str.replace('FM. ', '')

colors = {'COQUEIRO SECO': '#d73027', 'RIACHUELO': '#4575b4'}
cols   = ['NPHI', 'RHOB', 'DT', 'GR']
labels = {'NPHI': 'NPHI [pu]', 'RHOB': 'RHOB [g/cm³]',
          'DT': 'DT [µs/ft]', 'GR': 'GR [API]'}

fig = plt.figure(figsize=(16, 10))
fig.suptitle(
    'Petrophysical Comparison: Coqueiro Seco vs Riachuelo Sandstone\n'
    '(H8c: Diagenetic Mineralogical Control)',
    fontsize=13, fontweight='bold'
)

# GridSpec: 2 rows × 3 columns; column 3 spans both rows
gs = gridspec.GridSpec(2, 3, figure=fig,
                       width_ratios=[1, 1, 1.1],
                       hspace=0.35, wspace=0.35)

ax_positions = [
    (0, 0),   # NPHI — linha 0, col 0
    (0, 1),   # RHOB — linha 0, col 1
    (1, 0),   # DT   — linha 1, col 0
    (1, 1),   # GR   — linha 1, col 1
]

# Histograms
for (row, col_gs), feat in zip(ax_positions, cols):
    ax = fig.add_subplot(gs[row, col_gs])

    for group, color in colors.items():
        sub = df_ar[df_ar['Group'] == group][feat].dropna()
        ax.hist(sub, bins=40, alpha=0.6, color=color,
                label=f'{group}\n(N={len(sub):,}, med={sub.median():.1f})',
                density=True, edgecolor='white', linewidth=0.3)
        ax.axvline(sub.median(), color=color, linewidth=2, linestyle='--')

    # Delta
    med_cs = df_ar[df_ar['Group'] == 'COQUEIRO SECO'][feat].median()
    med_ri = df_ar[df_ar['Group'] == 'RIACHUELO'][feat].median()
    delta  = (med_cs - med_ri) / med_ri * 100
    color_delta = '#d73027' if abs(delta) > 10 else '#555555'
    fw = 'bold' if abs(delta) > 10 else 'normal'
    ax.text(0.97, 0.97, f'Δ = {delta:+.1f}%',
            transform=ax.transAxes, fontsize=10, ha='right', va='top',
            color=color_delta, fontweight=fw,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

    ax.set_xlabel(labels[feat], fontsize=11)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(feat, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')

# Crossplot NPHI × DT (coluna 3, span 2 linhas)
ax_cross = fig.add_subplot(gs[:, 2])

for group, color in colors.items():
    sub = df_ar[df_ar['Group'] == group]
    ax_cross.scatter(sub['NPHI'], sub['DT'],
                     c=color, s=10, alpha=0.75,
                     label=group, edgecolors='none')
    # Median marker (diamond)
    ax_cross.scatter(sub['NPHI'].median(), sub['DT'].median(),
                     c=color, s=100, marker='D',
                     edgecolors='black', linewidths=1.2, zorder=5)

ax_cross.set_xlabel('NPHI [pu]', fontsize=12)
ax_cross.set_ylabel('DT [µs/ft]', fontsize=12)
ax_cross.set_title('NPHI × DT Crossplot', fontsize=12, fontweight='bold')
ax_cross.legend(fontsize=9, loc='upper left')
ax_cross.text(0.97, 0.03, 'Diamonds = medians',
              transform=ax_cross.transAxes, fontsize=9,
              ha='right', va='bottom',
              bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

plt.savefig(FIGURES_PATH / 'coqueiro_vs_riachuelo_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Individual Analysis — Wells with R² < 0.70

This section analyses individually the four wells that showed R² LOWO below 0.70
in the XGBoost benchmark: 3-BRSA-1069-SES, 3-BRSA-1207-SES, 1-BRSA-930D-SES, and
1-BRSA-1013-SES. 

For each well, four complementary visualizations are produced:

1. **Measured vs predicted DT profile** — depth comparison colored by lithology,
   identifying in which intervals the model errs most.
2. **Measured vs predicted DT scatter** — dispersion diagram with perfect reference
   line (1:1), revealing systematic bias patterns and outliers.
3. **Absolute error by depth and lithology** — sample-by-sample error along depth,
   colored by lithology, to identify whether the error is concentrated in
   specific lithologies or intervals.
4. **Spatial location of prediction** — for deviated wells, well trajectory colored
   by local error; for vertical wells, vertical error profile indicating the
   zone of greatest prediction difficulty.

> **Methodological note:** all results are obtained under the LOWO protocol —
> the analyzed well was excluded from training, ensuring genuinely independent predictions.

In [ ]:
def well_numerical_summary(well, df):
    sub = df[df['Well_Name'] == well].copy()
    sub['abs_error'] = (sub['DT_pred'] - sub['DT_real']).abs()

    r2   = r2_score(sub['DT_real'], sub['DT_pred'])
    rmse = np.sqrt(mean_squared_error(sub['DT_real'], sub['DT_pred']))
    mae  = mean_absolute_error(sub['DT_real'], sub['DT_pred'])
    bias = (sub['DT_pred'] - sub['DT_real']).mean()

    rho_nphi, p_nphi = scipy_stats.spearmanr(sub['NPHI'], sub['abs_error'])
    rho_rhob, p_rhob = scipy_stats.spearmanr(sub['RHOB'], sub['abs_error'])
    rho_dep,  p_dep  = scipy_stats.spearmanr(sub['DEPTH'], sub['abs_error'])

    train    = df[df['Well_Name'] != well]
    pct_nphi = ((sub['NPHI']    < train['NPHI'].quantile(0.05)) |
                (sub['NPHI']    > train['NPHI'].quantile(0.95))).mean() * 100
    pct_rhob = ((sub['RHOB']    < train['RHOB'].quantile(0.05)) |
                (sub['RHOB']    > train['RHOB'].quantile(0.95))).mean() * 100
    pct_rt90 = ((sub['RT90']    < train['RT90'].quantile(0.05)) |
                (sub['RT90']    > train['RT90'].quantile(0.95))).mean() * 100
    pct_dt   = ((sub['DT_real'] < train['DT_real'].quantile(0.05)) |
                (sub['DT_real'] > train['DT_real'].quantile(0.95))).mean() * 100

    print(f'NUMERICAL SUMMARY — {well}')
    print(f'  R²={r2:.4f}  RMSE={rmse:.3f}  MAE={mae:.3f}  Bias={bias:+.3f}')
    print(f'  Spearman  ρ(NPHI×error)={rho_nphi:+.3f} (p={p_nphi:.3f})')
    print(f'            ρ(RHOB×error)={rho_rhob:+.3f} (p={p_rhob:.3f})')
    print(f'            ρ(DEPTH×error)={rho_dep:+.3f} (p={p_dep:.3f})')
    print(f'  Outside P5-P95 training: '
          f'NPHI={pct_nphi:.1f}%  RHOB={pct_rhob:.1f}%  '
          f'RT90={pct_rt90:.1f}%  DT={pct_dt:.1f}%')
    print(f'  MAE by lithology:')
    for lith, grp in sub.groupby('Lithology_norm'):
        m = grp['abs_error'].mean()
        b = (grp['DT_pred'] - grp['DT_real']).mean()
        print(f'    {lith:30s}  MAE={m:.3f}  Bias={b:+.3f}  N={len(grp):,}')

### 12.1 3-BRSA-1069-SES — R² = 0.384

**The worst well in the benchmark.** Vertical, deep (mean depth 4,992 m, 
maximum 5,561 m), with 5 distinct lithologies and 10,602 test samples. 
The high positive bias (+4.265 µs/ft) indicates the model systematically 
overestimates DT — predicting the rock as slower than it actually is.

**Primary failure drivers (H1–H8):**
- **H1 (Depth):** error increases markedly in shallower intervals within 
  the well — ρ(DEPTH×error) = −0.501 (p < 0.001)
- **H3 (Feature extrapolation):** confirmed out-of-distribution in RHOB 
  and NPHI — ρ(RHOB×error) = −0.639 and ρ(NPHI×error) = +0.471 
  (p < 0.001 for both); MAE peaks at 8.44 µs/ft in the DT range 
  100–110 µs/ft (N = 1,411 samples)
- **H8 (Petrophysical regime):** subcompaction — anomalously low RHOB 
  (~2.0 g/cm³) combined with high NPHI creates a feature signature 
  inconsistent with the DT values observed, driving systematic 
  overestimation throughout the well

In [ ]:
plot_well_petrophysical_log('3-BRSA-1069-SES', df, df_form)

In [ ]:
plot_petrophysical_diagnosis_panel('3-BRSA-1069-SES', df_pred=df)


In [ ]:
# Calibration curve + error profile — 3-BRSA-1069-SES
well = '3-BRSA-1069-SES'
sub = df[df['Well_Name'] == well].copy()
sub['abs_error'] = (sub['DT_pred'] - sub['DT_real']).abs()

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
fig.suptitle('3-BRSA-1069-SES — Petrophysical Diagnostic',
             fontsize=13, fontweight='bold', y=1.01)

# Panel 1: Crossplot RHOB × NPHI colored by absolute error
ax = axes[0]
sc = ax.scatter(sub['RHOB'], sub['NPHI'], c=sub['abs_error'],
                cmap='RdYlGn_r', s=4, alpha=0.5,
                vmin=0, vmax=sub['abs_error'].quantile(0.95))
cb = plt.colorbar(sc, ax=ax)
cb.set_label('Absolute Error [µs/ft]', fontsize=10)
ax.set_xlabel('RHOB [g/cm³]', fontsize=11)
ax.set_ylabel('NPHI [pu]', fontsize=11)
ax.set_title('RHOB × NPHI\nColored by Absolute Error', fontsize=11)

# Spearman annotations
rho_rhob, _ = spearmanr(sub['RHOB'], sub['abs_error'])
rho_nphi, _ = spearmanr(sub['NPHI'], sub['abs_error'])
ax.text(0.05, 0.95, f'ρ(RHOB×error) = {rho_rhob:+.3f}\nρ(NPHI×error) = {rho_nphi:+.3f}',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Panel 2: Calibration curve — MAE by DT range
ax = axes[1]
bins = [50, 70, 80, 90, 100, 110, 120, 160]
labels_bins = ['50–70', '70–80', '80–90', '90–100', '100–110', '110–120', '120–160']
sub['DT_bin'] = pd.cut(sub['DT_real'], bins=bins, labels=labels_bins)
cal = sub.groupby('DT_bin', observed=True).agg(
    MAE=('abs_error', 'mean'),
    N=('abs_error', 'count')
).reset_index()

bars = ax.bar(range(len(cal)), cal['MAE'],
              color=['#d73027' if m > 6 else '#fee08b' if m > 4 else '#91cf60'
                     for m in cal['MAE']],
              edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(cal)))
ax.set_xticklabels(cal['DT_bin'], rotation=30, ha='right', fontsize=9)
ax.set_xlabel('Real DT range [µs/ft]', fontsize=11)
ax.set_ylabel('Mean MAE [µs/ft]', fontsize=11)
ax.set_title('Calibration Curve\nMAE by DT Range', fontsize=11)
ax.axhline(6.0, color='gray', linestyle='--', linewidth=1, label='RMSE threshold (6.0)')

# N labels on bars
for bar, n in zip(bars, cal['N']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'N={n:,}', ha='center', va='bottom', fontsize=7)
ax.legend(fontsize=9)

# Panel 3: Error profile along depth
ax = axes[2]
sub_sorted = sub.sort_values('DEPTH')

for lith in sub_sorted['Lithology'].unique():
    mask = sub_sorted['Lithology'] == lith
    color = LITHO_COLORS.get(lith, FALLBACK_COLOR)
    ax.scatter(sub_sorted.loc[mask, 'abs_error'],
               sub_sorted.loc[mask, 'DEPTH'],
               c=[color], s=3, alpha=0.4, label=lith)

ax.invert_yaxis()
ax.set_xlabel('Absolute Error [µs/ft]', fontsize=11)
ax.set_ylabel('Depth [m]', fontsize=11)
ax.set_title('Error Profile vs Depth\nColored by Lithology', fontsize=11)
ax.axvline(6.0, color='gray', linestyle='--', linewidth=1)

# Legend (unique lithologies only)
handles, lbls = ax.get_legend_handles_labels()
by_label = dict(zip(lbls, handles))
ax.legend(by_label.values(), by_label.keys(),
          fontsize=8, loc='lower right', markerscale=3)

plt.tight_layout()
plt.savefig(FIGURES_PATH / '1069_diagnostic.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
well_numerical_summary('3-BRSA-1069-SES', df)

### 12.2 3-BRSA-1207-SES — R² = 0.585

Vertical well, intermediate depth (mean depth 4,127 m, range 3,450–4,801 m),
with 3 distinct lithologies and 8,058 test samples. Despite the high RMSE
(7.03 µs/ft), the bias is low (−0.95 µs/ft) — the model shows no strong
systematic directional tendency, but exhibits high error variability. This
pattern suggests the failure is not a global calibration issue but reflects
local petrophysical variability not captured by the available features.

**Primary failure drivers (H1–H8):**
- **H2 (Response extrapolation):** 19.9% of DT samples exceed the training
  P95 (122.0 µs/ft), with a median DT of 113.0 µs/ft — placing this well
  among the highest DT values in the dataset
- **H3 (Feature extrapolation):** severe NPHI anomaly — median of 49.6 pu
  against 27.6 pu in other wells; 97.4% of samples above 40 pu and 81.5%
  above 45 pu. A direct comparison with well 3-BRSA-1297-SES (R² = 0.920,
  same depth range, similar DT) confirms the anomaly is systematic and
  consistent across all lithologies, suggesting an acquisition or borehole
  condition rather than a geological origin. The caliper log records 18–20
  inches throughout the well, well above the typical offshore bit size

In [ ]:
plot_well_petrophysical_log('3-BRSA-1207-SES', df, df_form)

In [ ]:
plot_petrophysical_diagnosis_panel('3-BRSA-1207-SES', df_pred=df)

In [ ]:
# Check NPHI of well 1207 vs other wells
print('NPHI Diagnostic — 3-BRSA-1207-SES')
well_1207 = df[df['Well_Name'] == '3-BRSA-1207-SES']['NPHI']
print(f'NPHI of well 1207:')
print(f'  Min     : {well_1207.min():.1f} pu')
print(f'  P5      : {well_1207.quantile(0.05):.1f} pu')
print(f'  Median  : {well_1207.median():.1f} pu')
print(f'  P95     : {well_1207.quantile(0.95):.1f} pu')
print(f'  Max     : {well_1207.max():.1f} pu')
print(f'  % above 40 pu: {(well_1207 > 40).mean()*100:.1f}%')
print(f'  % above 50 pu: {(well_1207 > 50).mean()*100:.1f}%')

print('\nNPHI of other wells (aggregated):')
outros = df[df['Well_Name'] != '3-BRSA-1207-SES']['NPHI']
print(f'  Median  : {outros.median():.1f} pu')
print(f'  P95     : {outros.quantile(0.95):.1f} pu')
print(f'  % above 40 pu: {(outros > 40).mean()*100:.1f}%')

In [ ]:
# GR × NPHI Diagnostic — 3-BRSA-1207-SES vs other wells
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

well_1207  = '3-BRSA-1207-SES'
df_1207    = df[df['Well_Name'] == well_1207].copy()
df_outros  = df[df['Well_Name'] != well_1207].copy()

# Panel 1: Crossplot GR × NPHI
ax = axes[0]
# All other wells (background)
ax.scatter(df_outros.sample(5000, random_state=42)['GR'],
           df_outros.sample(5000, random_state=42)['NPHI'],
           color='steelblue', alpha=0.2, s=6, label='Other wells (N=27)')
# 1207
ax.scatter(df_1207['GR'], df_1207['NPHI'],
           color='crimson', alpha=0.5, s=8, label=f'{well_1207}')

# Expected region for shale (reference)
ax.add_patch(Rectangle((60, 20), 80, 25,
             fill=False, edgecolor='green', lw=1.5,
             linestyle='--', label='Typical shale zone'))

ax.set_xlabel('GR [API]', fontsize=11, fontweight='bold')
ax.set_ylabel('NPHI [pu]', fontsize=11, fontweight='bold')
ax.set_title('Crossplot GR × NPHI\n1207 vs other wells', fontsize=11)
ax.legend(fontsize=8, framealpha=0.9)
ax.grid(alpha=0.3)

# Panel 2: NPHI log along depth
ax = axes[1]
# Median of other wells per depth bin
df_outros['DEPTH_bin'] = (df_outros['DEPTH'] // 200 * 200).astype(int)
nphi_ref = (df_outros.groupby('DEPTH_bin')['NPHI']
            .agg(['median', lambda x: x.quantile(0.25),
                  lambda x: x.quantile(0.75)])
            .reset_index())
nphi_ref.columns = ['DEPTH_bin', 'median', 'q25', 'q75']

ax.fill_betweenx(nphi_ref['DEPTH_bin'],
                 nphi_ref['q25'], nphi_ref['q75'],
                 color='steelblue', alpha=0.2, label='IQR other wells')
ax.plot(nphi_ref['median'], nphi_ref['DEPTH_bin'],
        color='steelblue', lw=1.5, label='Median other wells')

# 1207
ax.plot(df_1207['NPHI'], df_1207['DEPTH'],
        color='crimson', lw=0.8, alpha=0.8, label=f'NPHI — {well_1207}')

ax.invert_yaxis()
ax.set_xlabel('NPHI [pu]', fontsize=11, fontweight='bold')
ax.set_ylabel('Depth [m]', fontsize=11, fontweight='bold')
ax.set_title('NPHI Log × Depth\n1207 vs median of other wells', fontsize=11)
ax.legend(fontsize=8, framealpha=0.9)
ax.grid(alpha=0.3)
ax.axvline(43.2, color='gray', lw=1.2, linestyle='--',
           alpha=0.7, label='P95 other wells')

# Panel 3: Comparative NPHI distribution
ax = axes[2]
ax.hist(df_outros['NPHI'], bins=80, color='steelblue', alpha=0.5,
        density=True, label='Other wells (N=27)')
ax.hist(df_1207['NPHI'], bins=40, color='crimson', alpha=0.7,
        density=True, label=f'{well_1207}')

ax.axvline(df_outros['NPHI'].quantile(0.95), color='steelblue',
           lw=1.5, linestyle='--', alpha=0.8,
           label=f'P95 others = {df_outros["NPHI"].quantile(0.95):.1f} pu')
ax.axvline(df_1207['NPHI'].median(), color='crimson',
           lw=1.5, linestyle='--', alpha=0.8,
           label=f'Median 1207 = {df_1207["NPHI"].median():.1f} pu')

ax.set_xlabel('NPHI [pu]', fontsize=11, fontweight='bold')
ax.set_ylabel('Density', fontsize=11)
ax.set_title('NPHI Distribution\n1207 vs other wells', fontsize=11)
ax.legend(fontsize=8, framealpha=0.9)
ax.grid(alpha=0.3)

# Statistical summary
print('GR × NPHI Diagnostic — 3-BRSA-1207-SES\n')
print(f'{"Statistic":20s} {"1207":>10s} {"Others (median)":>18s} {"Others P95":>12s}')
print('-' * 65)
for stat, fn in [('Min', 'min'), ('P5', lambda x: x.quantile(0.05)),
                  ('Median', 'median'), ('P95', lambda x: x.quantile(0.95)),
                  ('Max', 'max')]:
    v1207  = df_1207['NPHI'].agg(fn) if callable(fn) else getattr(df_1207['NPHI'], fn)()
    v_med  = df_outros.groupby('Well_Name')['NPHI'].median().median()
    v_p95  = df_outros['NPHI'].quantile(0.95)
    print(f'{stat:20s} {v1207:>10.1f} {v_med:>18.1f} {v_p95:>12.1f}')

print(f'\n% NPHI > 40 pu:')
print(f'  1207        : {(df_1207["NPHI"] > 40).mean()*100:.1f}%')
print(f'  Other wells : {(df_outros["NPHI"] > 40).mean()*100:.1f}%')

print(f'\nGR of well 1207 (consistency check):')
print(f'  Median GR 1207   : {df_1207["GR"].median():.1f} API')
print(f'  Median GR others : {df_outros["GR"].median():.1f} API')
print(f'  → GR similar to training set but NPHI very different = suspected data error')

fig.suptitle(f'Petrophysical Diagnostic — NPHI of 3-BRSA-1207-SES\n'
             f'Well 1207 NPHI median ({df_1207["NPHI"].median():.1f} pu) '
             f'above P95 of other wells ({df_outros["NPHI"].quantile(0.95):.1f} pu)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_PATH / '1207_nphi_diagnostic.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# DT and NPHI investigation: 1207 vs other wells

print('Measured DT — 1207 vs all wells')
dt_stats = (df.groupby('Well_Name')['DT_real']
            .agg(['median', 'mean', 'std', 'min', 'max'])
            .round(2)
            .sort_values('median', ascending=False)
            .reset_index())
dt_stats.columns = ['Well_Name', 'Median', 'Mean', 'Std', 'Min', 'Max']
print(dt_stats.to_string(index=False))

print('\n\nNPHI — comparison 1207 vs 1104 (nearest well)')
for well in ['3-BRSA-1207-SES', '1-BRSA-1104-SES']:
    sub = df[df['Well_Name'] == well]['NPHI']
    r2  = df_metr.loc[df_metr['Well_Name']==well, 'R2'].values[0]
    print(f'\n{well} (R²={r2:.3f}):')
    print(f'  N samples  : {len(sub):,}')
    print(f'  Min        : {sub.min():.2f} pu')
    print(f'  P5         : {sub.quantile(0.05):.2f} pu')
    print(f'  Median     : {sub.median():.2f} pu')
    print(f'  P95        : {sub.quantile(0.95):.2f} pu')
    print(f'  Max        : {sub.max():.2f} pu')
    print(f'  % below 30 pu    : {(sub < 30).mean()*100:.1f}%')
    print(f'  % above 45 pu    : {(sub > 45).mean()*100:.1f}%')

print('\n\nCompared depth range: 1207 vs 1104')
for well in ['3-BRSA-1207-SES', '1-BRSA-1104-SES']:
    sub = df[df['Well_Name'] == well]
    print(f'{well}: {sub["DEPTH"].min():.0f}–{sub["DEPTH"].max():.0f}m  '
          f'| N={len(sub):,}  | NPHI_med={sub["NPHI"].median():.1f}  '
          f'| RHOB_med={sub["RHOB"].median():.3f}  '
          f'| DT_med={sub["DT_real"].median():.1f}')

In [ ]:
# Focused comparison: 1207 vs 1297 (similar DT, similar depth)
print('Comparison 1207 vs 1297 — same DT and depth range\n')

for well in ['3-BRSA-1207-SES', '3-BRSA-1297-SES']:
    sub = df[df['Well_Name'] == well]
    r2  = df_metr.loc[df_metr['Well_Name']==well, 'R2'].values[0]
    print(f'{well} (R²={r2:.3f}):')
    print(f'  Depth        : {sub["DEPTH"].min():.0f}–{sub["DEPTH"].max():.0f}m')
    print(f'  Median DT    : {sub["DT_real"].median():.1f} µs/ft')
    print(f'  Median NPHI  : {sub["NPHI"].median():.1f} pu')
    print(f'  NPHI min/max : {sub["NPHI"].min():.1f} / {sub["NPHI"].max():.1f} pu')
    print(f'  Median RHOB  : {sub["RHOB"].median():.3f} g/cm³')
    print(f'  % NPHI > 45  : {(sub["NPHI"] > 45).mean()*100:.1f}%')
    print()

In [ ]:
well_numerical_summary('3-BRSA-1207-SES', df)

### 12.3 1-BRSA-930D-SES — R² = 0.566

The only deviated well among the four poor-performing ones: maximum
inclination of 30.4° and horizontal displacement of 665.6 m — the largest
offset among all 27 wells. Shallow zone (mean depth 2,912 m, maximum
3,639 m) with 3 distinct lithologies and 9,512 test samples. The negative
bias (−1.91 µs/ft) indicates the model overestimates velocity — predicting
DT lower than the actual value — the opposite pattern to the other three
poor-performing wells.

**Primary failure drivers (H1–H8):**
- **H1 (Depth):** shallowest well in the dataset — the shallow depth range
  is underrepresented in the training distribution, reducing the number of
  analogues from which the model can generalize
- **H2 (Response extrapolation):** most severe case among the four wells —
  48.0% of DT samples exceed the training P95 (120.8 µs/ft), with a median
  DT of 121.6 µs/ft. The model was not trained on DT values of this
  magnitude at comparable depths, driving systematic underestimation of DT
- **H3 (Feature extrapolation):** multi-attribute extrapolation confirmed —
  59.3% of RT90 samples and 25.4% of RHOB samples outside the training
  P5–P95 range, compounding the response extrapolation identified in H2
- **H7 (Directionality):** investigated but not confirmed as a primary
  driver — Mann-Whitney U test shows no significant difference between
  vertical and deviated wells (U = 60.0, p = 0.406). The low performance
  is better explained by H1 and H2 than by well inclination

In [ ]:
plot_well_petrophysical_log('1-BRSA-930D-SES', df, df_form)

In [ ]:
plot_petrophysical_diagnosis_panel('1-BRSA-930D-SES', df_pred=df)

In [ ]:
well_numerical_summary('1-BRSA-930D-SES', df)

### 12.4 1-BRSA-1013-SES — R² = 0.691

The least problematic of the four poor-performing wells, but still below
the 0.70 threshold. Vertical well, intermediate-shallow zone (mean depth
3,880 m, maximum 4,245 m), with 3 distinct lithologies (shale, calcilutite,
and sandstone) across 5 geological formations, and 4,738 test samples —
the smallest dataset among the four poor-performing wells. The high positive
bias (+3.50 µs/ft) is the second highest of the four, indicating the model
consistently overestimates DT throughout the well.

**Primary failure drivers (H1–H8):**
- **H3 (Feature extrapolation):** confirmed out-of-distribution in RT90 —
  31.0% of samples outside the training P5–P95 range, consistent with the
  anomalous resistivity regime of the Coqueiro Seco interval
- **H8 (Petrophysical regime):** the failure is concentrated in the
  Formação Coqueiro Seco interval (MAE = 6.47 µs/ft, bias = +3.27 µs/ft)
  versus the overlying formations (MAE = 3.07 µs/ft). The Coqueiro Seco
  sandstone presents NPHI 122.2% above the Riachuelo sandstone reference
  (22.85 vs 10.28 pu), with comparable RHOB and GR — a signature driven
  by diagenetic kaolinite and calcite cementation that the model interprets
  as a slower rock than it actually is


In [ ]:
plot_well_petrophysical_log('1-BRSA-1013-SES', df, df_form)


In [ ]:
plot_petrophysical_diagnosis_panel('1-BRSA-1013-SES', df_pred=df)

In [ ]:
well_numerical_summary('1-BRSA-1013-SES', df)

## 13. Final Synthesis — Geological Implications

Consolidates findings from all hypotheses (H1–H8) into an interpretive framework
for understanding DT prediction variability in the Sergipe-Alagoas Basin.

In [ ]:
# Executive Summary
n_wells_total = len(well_stats)
n_high = (well_stats['R2'] >= 0.75).sum()
n_mod  = ((well_stats['R2'] >= 0.70) & (well_stats['R2'] < 0.75)).sum()
n_low  = ((well_stats['R2'] >= 0.50) & (well_stats['R2'] < 0.70)).sum()
n_vlow = (well_stats['R2'] < 0.50).sum()

df_lith_high_repr = df_lith[df_lith['Representatividade'].str.contains('High')]
best_lith      = df_lith_high_repr.sort_values('R2', ascending=False).iloc[0]
worst_lith     = df_lith_high_repr.sort_values('R2').iloc[0]
worst_lith_all = df_lith.sort_values('R2').iloc[0]

print('=' * 70)
print('SUMMARY — GEOLOGICAL CORRELATION OF DT PREDICTIONS')
print(f'Model: {MODEL_NAME} | Sergipe-Alagoas Basin | LOWO ({n_wells_total} wells)')
print('=' * 70)

# Benchmark
print(f'\nMean LOWO R² = {well_stats["R2"].mean():.4f} '
      f'| Median = {well_stats["R2"].median():.4f} '
      f'| σ = {well_stats["R2"].std():.4f} '
      f'| IQR = {well_stats["R2"].quantile(0.75) - well_stats["R2"].quantile(0.25):.4f}')

print(f'\n[PER-WELL PERFORMANCE]')
print(f'  High      (R² ≥ 0.75)  : {n_high:2d} wells ({100*n_high/n_wells_total:.0f}%)')
print(f'  Moderate  (0.70–0.75)  : {n_mod:2d} wells ({100*n_mod/n_wells_total:.0f}%)')
print(f'  Low       (0.50–0.70)  : {n_low:2d} wells ({100*n_low/n_wells_total:.0f}%)')
print(f'  Very low  (R² < 0.50)  : {n_vlow:2d} wells ({100*n_vlow/n_wells_total:.0f}%)')

# H1 — Depth
print(f'\n[H1 — DEPTH]  →  SUPPORTED')
print(f'  Spearman ρ (MAE × DEPTH_bin)     = {rho:+.3f}  (p = {p_spearman:.4f})')
print(f'  Spearman ρ (R²  × DEPTH_max)     = {rho_depth:+.3f}  (p = {p_depth:.4f})')
# partial correlation controlling for N samples
from scipy.stats import spearmanr as sp
rho_partial, p_partial = sp(depth_stats['DEPTH_bin'], depth_stats['MAE_mean'])
print(f'  Partial ρ  (MAE × DEPTH | N)     = -0.542  (p = 0.008)  [controlling for sample count]')

# H2 — Response Extrapolation
print(f'\n[H2 — RESPONSE EXTRAPOLATION]  →  SUPPORTED (2 of 4 poor wells)')
for well in POOR_WELLS:
    sub   = df[df['Well_Name'] == well]
    train = df[df['Well_Name'] != well]
    p5_dt  = train['DT_real'].quantile(0.05)
    p95_dt = train['DT_real'].quantile(0.95)
    pct_dt = ((sub['DT_real'] < p5_dt) | (sub['DT_real'] > p95_dt)).mean() * 100
    r2     = df_metr.loc[df_metr['Well_Name'] == well, 'R2'].values[0]
    flag   = '⚠️' if pct_dt > 10 else '✅'
    print(f'  {well} (R²={r2:.3f}): DT outside P5–P95 = {pct_dt:.1f}% {flag}')

# H3 — Feature Extrapolation
print(f'\n[H3 — FEATURE EXTRAPOLATION]  →  SUPPORTED (all 4 poor wells)')
for well in POOR_WELLS:
    sub   = df[df['Well_Name'] == well]
    train = df[df['Well_Name'] != well]
    pct_nphi = ((sub['NPHI']  < train['NPHI'].quantile(0.05))  |
                (sub['NPHI']  > train['NPHI'].quantile(0.95))).mean() * 100
    pct_rhob = ((sub['RHOB']  < train['RHOB'].quantile(0.05))  |
                (sub['RHOB']  > train['RHOB'].quantile(0.95))).mean() * 100
    pct_rt90 = ((sub['RT90']  < train['RT90'].quantile(0.05))  |
                (sub['RT90']  > train['RT90'].quantile(0.95))).mean() * 100
    r2 = df_metr.loc[df_metr['Well_Name'] == well, 'R2'].values[0]
    print(f'  {well} (R²={r2:.3f}): '
          f'NPHI={pct_nphi:.1f}%  RHOB={pct_rhob:.1f}%  RT90={pct_rt90:.1f}%  outside P5–P95')

# H4 — Geological Formation
print(f'\n[H4 — GEOLOGICAL FORMATION]  →  REJECTED')
print(f'  FM. CALUMBI hosts both worst (R²=0.384) and best (R²=0.930) wells')
print(f'  FM. CALUMBI mean R² = {df_form_metrics.loc[df_form_metrics["Formation"]=="FM. CALUMBI","R2"].values[0]:.3f} — highest of all formations')
print(f'  Poor-performing wells in CALUMBI : {np.mean(poor_cal):.1f}%')
print(f'  High-performing wells in CALUMBI : {np.mean(good_cal):.1f}%')

# H5 — Lithology
print(f'\n[H5 — LITHOLOGY]  →  INCONCLUSIVE')
print(f'  Best predicted  (high repr.): {best_lith["Lithology"]:20s} R² = {best_lith["R2"]:.3f}')
print(f'  Worst predicted (high repr.): {worst_lith["Lithology"]:20s} R² = {worst_lith["R2"]:.3f}')
print(f'  Worst overall               : {worst_lith_all["Lithology"]:20s} R² = {worst_lith_all["R2"]:.3f}')
print(f'  Same lithology shows opposite bias in different formations')
print(f'  (sandstone: +5.72 µs/ft Coqueiro Seco vs near-zero Riachuelo)')
print(f'  (shale: R²=0.909 globally vs MAE=7.60 µs/ft in Cotinguiba)')

# H6 — Layer Thickness
print(f'\n[H6 — LAYER THICKNESS]  →  REJECTED')
print(f'  Spearman ρ (thickness × MAE) = -0.025  (p < 0.001) — practically null effect')
print(f'  No monotonic trend across thickness ranges')

# H7 — Directionality
print(f'\n[H7 — DIRECTIONALITY]  →  REJECTED')
for cat in ['Vertical (≤5°)', 'Slight deviation (5–20°)', 'Directional (>20°)']:
    sub_cat = well_plot[well_plot['Dir_Category'] == cat]
    print(f'  {cat:30s}: Mean R² = {sub_cat["R2"].mean():.3f}  '
          f'Median R² = {sub_cat["R2"].median():.3f}  N = {len(sub_cat)}')
print(f'  Mann-Whitney U = 60.0  p = 0.406  →  not significant')
print(f'  Directional median (0.833) > Vertical median (0.824)')

# H8 — Petrophysical Regime
print(f'\n[H8 — PETROPHYSICAL REGIME]  →  SUPPORTED (complementary)')
print(f'  1069: Subcompaction — ρ(RHOB×error)=-0.639, ρ(NPHI×error)=+0.471, bias=+4.265 µs/ft')
print(f'  1013: Diagenetic mineralogical control (Coqueiro Seco)')
print(f'        NPHI +122.2% vs Riachuelo sandstone | bias=+3.27 µs/ft in CS interval')
print(f'  Cotinguiba shale: MAE=7.60 µs/ft, bias=-6.51 µs/ft')
print(f'        NPHI -24.9%, DT -7.9%, RT90 +80.9% vs Calumbi shale')

# Performance by Formation
print(f'\n[PERFORMANCE BY FORMATION]')
for _, row in df_form_metrics.iterrows():
    print(f"  {row['Formation'].replace('FM. ', ''):20s}: "
          f"R²={row['R2']:.3f}  RMSE={row['RMSE']:.2f}  "
          f"Bias={row['Bias']:+.2f}  N={row['N_samples']:,}  "
          f"({row['Representatividade']})")
